<a href="https://colab.research.google.com/github/Lagnajit09/100x_AI_ML/blob/main/calibrated-typed-decision/TrainACalibratedModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **[1] — Install**

In [1]:
!pip install -q "transformers>=4.48" datasets peft   # peft: LoRA adapters
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 71.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## **[2] — Imports and GPU check**

In [2]:
import os, json, math, random, time, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__, "| transformers", transformers.__version__)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found: Runtime > Change runtime type > T4 GPU")

device: cuda | torch 2.14.1+cu130 | transformers 5.22.1
GPU: Tesla T4


## **[3] — Google Drive**

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## **[4] — Settings**

In [21]:
SMOKE = False     # True: tiny practice run (a few minutes). False: the real experiment.
SEED  = 2        # Step 2: train with SEED = 1, then SEED = 2. Seed 0 is the run you already have.
RERUN = False    # Step 7: True to retrain seeds 1-2 into NEW folders (seed1_rerun, seed2_rerun), with weights

MODEL_SIZE   = "base"            # Step 8: "large" trains ModernBERT-large (targets E and F only)
ADAPT        = "full"            # Step 9: "lora" = ModernBERT-large with LoRA adapters (target F only)
MODEL_NAME   = f"answerdotai/ModernBERT-{MODEL_SIZE}"
DATASET_NAME = "m-lagnajit/calibrated-typed-decisions"
DATASET_REV  = "main"           # v2. v1 is still available as revision="v1.0"
# A NEW folder: the v1 folder holds finished v1 runs, which Cell 15 would otherwise load
BASE_DIR = "/content/drive/MyDrive/ctd_student_v2_smoke" if SMOKE else "/content/drive/MyDrive/ctd_student_v2"
assert not (RERUN and SEED == 0), "seed 0 isn't retrained: its weights already exist"
OUT_DIR  = (BASE_DIR if SEED == 0 else                              # seed 0 stays exactly where it was
            f"{BASE_DIR}/seed{SEED}_rerun" if RERUN else f"{BASE_DIR}/seed{SEED}")
assert ADAPT in ("full", "lora") and not (ADAPT == "lora" and MODEL_SIZE == "base"), \
    "Step 9 uses LoRA on the large model only"
if MODEL_SIZE == "large":
    assert not RERUN, "RERUN is for the base model only"
    OUT_DIR = f"{BASE_DIR}/{'large_lora' if ADAPT == 'lora' else 'large'}/seed{SEED}"   # never touches base
os.makedirs(OUT_DIR, exist_ok=True)
SEEDS_ALL = [0, 1, 2]

RUNS = ["A", "B", "C", "D", "E", "F"]
# A: right answers only            B: right answers, softened evenly
# C: the dataset's soft_label      D: a blend of A and C
# E: C, corrected wherever the teacher's top pick was wrong
# F: a blend of A and E (the human answer + the corrected label)
COMPARISONS = ([(c, b) for c in ["C", "D", "E", "F"] for b in ["A", "B"]]
               + [("F", "D"), ("F", "E"), ("F", "C"), ("E", "D"), ("E", "C")])
SPRINT_RUNS = ["W25", "W75"]                 # Sprint: e.g. ["G", "H"], later ["W25", "W75"]; trains only these, keeps weights
assert not SPRINT_RUNS or (MODEL_SIZE == "base" and not RERUN), "sprint runs use the base model and the original seeds"
TRAIN_RUNS  = SPRINT_RUNS if SPRINT_RUNS else (RUNS if MODEL_SIZE == "base" else
               ["F"] if ADAPT == "lora" else ["E", "F"])         # Step 8: large E, F; Step 9: LoRA F

# Step 2: building the input
MAX_LEN        = 512            # longest sequence the student reads
MAX_Q_TOKENS   = 96             # v2: yes/no questions carry a description, PubMedQA a research question
MAX_OPT_TOKENS = 32             # v2: each option is "label: description"

# Step 6: training
BATCH_SIZE   = 16
MICRO_BATCH  = 8 if MODEL_SIZE == "large" else 16   # large (full or LoRA): 2 pieces of 8 = one step of 16
GRAD_CKPT    = False            # Step 8: set True only if the large model runs out of GPU memory (not with LoRA)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05     # Step 9: standard settings, fixed before training
LORA_TARGETS = ["Wqkv", "Wo", "Wi"]                 # attention (Wqkv, Wo) and feed-forward (Wi, Wo) layers
LR_LORA      = 2e-4             # Step 9: adapters are disposable, so bigger steps (your LoRA learning-rate row)
EPOCHS       = 1 if SMOKE else 3
LR_BODY      = 3e-5             # small, careful steps for the pre-trained body
LR_HEAD      = 1e-3             # bigger steps for the brand-new scorer
WEIGHT_DECAY = 0.01
WARMUP_FRAC  = 0.10
CLIP_NORM    = 1.0
SMOOTHING    = 0.1              # Run B: how much belief to spread evenly
BLEND        = 0.5              # Run D: share of the target given to the human answer
USE_AMP      = True             # 16-bit math on the GPU; set False if the loss turns NaN
SAVE_WEIGHTS = (not SMOKE) and ((MODEL_SIZE == "base" and (SEED == 0 or RERUN)) or
                                (MODEL_SIZE == "large" and (SEED == 0 or ADAPT == "lora")))
SAVE_WEIGHTS = SAVE_WEIGHTS or ((not SMOKE) and bool(SPRINT_RUNS))   # sprint: keep 16-bit weights for every seed
# large full fine-tuning: seed 0 only (Drive space); LoRA: every seed, adapters + scorer only (tens of MB)

# Step 7: evaluation
N_BINS       = 10
N_BOOT       = 1000
MIN_FIT_ROWS = 30
DIAL_SETUP   = "2 dials"        # Step 1's decision. "12 dials" = one per source and type (the old setup)
SMOKE_SIZES  = {"train": 160, "validation": 80, "test": 80, "heldout": 40}

def amp_on():
    return USE_AMP and device == "cuda"

## **[5] — Load your dataset**

In [5]:
ds = load_dataset(DATASET_NAME, revision=DATASET_REV)
print(ds)
assert "heldout" in ds and "soft_label" in ds["train"].column_names, "this notebook needs v2 of the dataset"

def take(split):
    rows = list(ds[split])
    if SMOKE:
        rows = random.Random(SEED).sample(rows, min(SMOKE_SIZES[split], len(rows)))
    return rows

DATA = {split: take(split) for split in ["train", "validation", "test", "heldout"]}
print({split: len(rows) for split, rows in DATA.items()})
print(pd.DataFrame(DATA["train"]).groupby(["source", "type"]).size())
print("\ntraining targets come from:", pd.Series([r["soft_label_source"] for r in DATA["train"]]).value_counts().to_dict())

r = next(r for r in DATA["train"] if r["type"] == "choice")
print(json.dumps({k: r[k] for k in ["source", "type", "instructions", "state", "options",
                                    "option_descriptions", "gold", "soft_label", "soft_label_source"]},
                 indent=2, ensure_ascii=False)[:1400])

README.md:   0%|          | 0.00/20.6k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 8.15MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  889kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  898kB            

... (4 outputs left out)

Generating validation split:   0%|          | 0/1799 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1797 [00:00<?, ? examples/s]

Generating heldout split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source', 'corrected_soft_label', 'soft_label_corrected'],
        num_rows: 16995
    })
    validation: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'instructions', 'state', 'options', 'option_descriptions', 'candidate', 'gold', 'gold_index', 'display_perm', 'teacher_option_logprobs', 'teacher_temperature', 'teacher_shift', 'teacher_probs', 'teacher_letter_mass', 'human_probs', 'soft_label', 'soft_label_source', 'corrected_soft_label', 'soft_label_corrected'],
        num_rows: 1799
    })
    test: Dataset({
        features: ['id', 'source', 'source_license', 'source_id', 'type', 'ins

## **[6] — Tokenizer and special tokens**

In [6]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
CLS, SEP, MASK, PAD = tok.cls_token_id, tok.sep_token_id, tok.mask_token_id, tok.pad_token_id
print({"[CLS]": CLS, "[SEP]": SEP, "[MASK]": MASK, "[PAD]": PAD})
assert None not in (CLS, SEP, MASK, PAD), "tokenizer is missing a special token"

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

{'[CLS]': 50281, '[SEP]': 50282, '[MASK]': 50284, '[PAD]': 50283}


## **[7] — Tokenize every piece once**

In [7]:
def option_texts(row):
    """What the student reads for each option, in stored order. v2: 'label: description'."""
    if row["type"] == "noul":
        return ["no", "yes"]
    descs = row.get("option_descriptions") or [""] * len(row["options"])
    return [f"{o}: {d}" if d else o for o, d in zip(row["options"], descs)]

def pretokenize(rows):
    qs   = tok([r["instructions"] for r in rows], add_special_tokens=False)["input_ids"]
    txts = tok([" " + r["state"] for r in rows], add_special_tokens=False)["input_ids"]
    flat = [" " + o for r in rows for o in option_texts(r)]
    flat_ids = tok(flat, add_special_tokens=False)["input_ids"]
    k = 0
    for r, q, t in zip(rows, qs, txts):
        n = len(option_texts(r))
        r["q_ids"]   = q[:MAX_Q_TOKENS]
        r["opt_ids"] = [o[:MAX_OPT_TOKENS] for o in flat_ids[k:k + n]]
        r["txt_ids"] = t
        k += n

for split in DATA:
    pretokenize(DATA[split])

r = next(r for r in DATA["train"] if r["type"] == "choice")
print("question pieces:", tok.convert_ids_to_tokens(r["q_ids"]))
print("option pieces:  ", [tok.convert_ids_to_tokens(o) for o in r["opt_ids"]])

# How much room is left for the text, in the worst row of every split?
head = max(2 + len(r["q_ids"]) + sum(1 + len(o) for o in r["opt_ids"]) + 1
           for split in DATA for r in DATA[split])
print(f"longest question + options: {head} tokens -> the text always keeps at least {MAX_LEN - head}")

question pieces: ['Which', 'Ġintent', 'Ġbest', 'Ġdescribes', 'Ġthis', 'Ġrequest', 'Ġto', 'Ġa', 'Ġvirtual', 'Ġassistant', '?']
option pieces:   [['Ġmeeting', 'Ġschedule', ':', 'Ġasking', 'Ġabout', 'Ġscheduled', 'Ġmeetings', 'Ġand', 'Ġtheir', 'Ġtim', 'ings'], ['Ġroll', 'over', 'Ġ401', 'k', ':', 'Ġroll', 'over', 'Ġof', 'Ġ401', 'k', 'Ġfunds'], ['Ġtaxes', ':', 'Ġasking', 'Ġabout', 'Ġtax', 'Ġpayments', 'Ġand', 'Ġobligations'], ['Ġspending', 'Ġhistory', ':', 'Ġasking', 'Ġabout', 'Ġpast', 'Ġspending'], ['Ġaccount', 'Ġblocked', ':', 'Ġaccount', 'Ġaccess', 'Ġor', 'Ġtransaction', 'Ġfunctionality', 'Ġrestricted'], ['Ġrestaurant', 'Ġsuggestion', ':', 'Ġsuggestions', 'Ġfor', 'Ġrestaurants']]
longest question + options: 122 tokens -> the text always keeps at least 390


## **[8] — Build one input sequence**

In [8]:
def build_input(row, rng=None):
    """[CLS] question [SEP] [MASK] opt [MASK] opt ... [SEP] text [SEP]
    rng=None -> options in stored order (evaluation)
    rng      -> options in a random order (training), so position can't become a shortcut"""
    n = len(row["opt_ids"])
    perm = list(range(n))
    if rng is not None:
        rng.shuffle(perm)                  # shown slot d holds stored option perm[d]
    ids = [CLS] + row["q_ids"] + [SEP]
    marker_pos = []
    for c in perm:
        marker_pos.append(len(ids))        # this option's [MASK] lands here
        ids += [MASK] + row["opt_ids"][c]
    ids += [SEP]
    room = MAX_LEN - len(ids) - 1          # leave space for the final [SEP]
    ids += row["txt_ids"][:room] + [SEP]   # only the text can ever be cut
    return {"input_ids": ids, "marker_pos": marker_pos, "perm": perm}

ex = build_input(DATA["train"][0])
print(len(ex["input_ids"]), "tokens | markers at", ex["marker_pos"])
for p, piece in enumerate(tok.convert_ids_to_tokens(ex["input_ids"])[:40]):
    print(f"{p:3d}  {piece}{'   <- marker' if p in ex['marker_pos'] else ''}")

81 tokens | markers at [13, 25, 37, 46, 54, 64]
  0  [CLS]
  1  Which
  2  Ġintent
  3  Ġbest
  4  Ġdescribes
  5  Ġthis
  6  Ġrequest
  7  Ġto
  8  Ġa
  9  Ġvirtual
 10  Ġassistant
 11  ?
 12  [SEP]
 13  [MASK]   <- marker
 14  Ġmeeting
 15  Ġschedule
 16  :
 17  Ġasking
 18  Ġabout
 19  Ġscheduled
 20  Ġmeetings
 21  Ġand
 22  Ġtheir
 23  Ġtim
 24  ings
 25  [MASK]   <- marker
 26  Ġroll
 27  over
 28  Ġ401
 29  k
 30  :
 31  Ġroll
 32  over
 33  Ġof
 34  Ġ401
 35  k
 36  Ġfunds
 37  [MASK]   <- marker
 38  Ġtaxes
 39  :


## **[9] — Each run’s target**

In [9]:
def make_target(row, run):
    """The run's target, in stored order."""
    n, g = len(row["opt_ids"]), row["gold_index"]
    onehot = np.zeros(n)
    onehot[g] = 1.0
    if run == "A":                                   # 100% on the right answer
        return onehot
    if run == "B":                                   # 0.9 + 0.1/n on the right answer, 0.1/n elsewhere
        t = np.full(n, SMOOTHING / n)
        t[g] += 1.0 - SMOOTHING
        return t
    soft = np.array(row["soft_label"], dtype=float)  # rater votes (GoEmotions) or calibrated teacher
    soft = soft / soft.sum()
    if run == "C":
        return soft
    if run == "D":                                   # part human answer, part soft label
        return BLEND * onehot + (1 - BLEND) * soft
    if run == "E":                                   # fix the teacher's wrong top picks, keep its shape
        if row["soft_label_source"] == "teacher":
            k = int(soft.argmax())
            if k != g:
                soft[k], soft[g] = soft[g], soft[k]  # the right answer gets the teacher's top value
        return soft
    if run == "F":                                   # part human answer, part corrected label (v2.1 column)
        corr = row.get("corrected_soft_label")
        corr = make_target(row, "E") if corr is None else np.array(corr, dtype=float) / np.sum(corr)
        return BLEND * onehot + (1 - BLEND) * corr
    if run in ("G", "H"):                            # Sprint controls: E's right-answer value, nothing else of E's
        e = make_target(row, "E")
        if row["type"] == "noul":                    # one wrong option: nothing to shuffle or spread
            return e
        wrong = [j for j in range(n) if j != g]
        t = e.copy()
        if run == "H":                               # E flattened: the rest spread evenly over the wrong options
            t[wrong] = (1.0 - e[g]) / len(wrong)
            return t
        import zlib                                  # E shuffled: same wrong-option values, in a random order
        vals = list(e[wrong])                        # fixed per question (seeded by its id), shared by all seeds
        random.Random(zlib.crc32(str(row["id"]).encode())).shuffle(vals)
        t[wrong] = vals
        return t
    if run.startswith("W"):                          # Sprint sweep: W25 = 25% human answer + 75% corrected label
        w = int(run[1:]) / 100
        return w * onehot + (1 - w) * make_target(row, "E")
    raise ValueError(run)

# Show every target on a row where the teacher's top pick was wrong
wrong = [r for r in DATA["train"]
         if r["soft_label_source"] == "teacher" and int(np.argmax(r["soft_label"])) != r["gold_index"]]
r = wrong[0] if wrong else DATA["train"][0]
print("options:", option_texts(r), "\nright answer:", r["gold"])
for run in RUNS:
    print(f"target {run}:", np.round(make_target(r, run), 3))
teacher_rows = sum(r["soft_label_source"] == "teacher" for r in DATA["train"])
print(f"\nRun E changes {len(wrong)} of {teacher_rows} teacher-labelled training rows "
      f"({len(wrong) / max(1, teacher_rows):.1%}); rater-labelled GoEmotions rows are left as they are")

options: ['timezone: time zones of locations around the world', 'pay bill: request to pay a bill', "change user name: change user's name in system settings", 'date: request for current or future dates', 'travel alert: travel advisories and safety warnings', 'directions: requests for travel or location directions', 'traffic: traffic conditions and forecasts', 'distance: distance travel time estimates'] 
right answer: distance
target A: [0. 0. 0. 0. 0. 0. 0. 1.]
target B: [0.012 0.012 0.012 0.012 0.012 0.012 0.012 0.912]
target C: [0.001 0.001 0.001 0.001 0.002 0.772 0.005 0.216]
target D: [0.001 0.001 0.001 0.001 0.001 0.386 0.002 0.608]
target E: [0.001 0.001 0.001 0.001 0.002 0.216 0.005 0.772]
target F: [0.001 0.001 0.001 0.001 0.001 0.108 0.002 0.886]

Run E changes 1391 of 13996 teacher-labelled training rows (9.9%); rater-labelled GoEmotions rows are left as they are


In [15]:
row = next(r for r in DATA["train"] if r["type"] == "choice" and r["soft_label_corrected"])
print("options:", row["options"], "| right answer:", row["options"][row["gold_index"]])
for run in ["E", "G", "H", "W25", "W75"]:
    print(run, np.round(make_target(row, run), 3))

options: ['timezone', 'pay bill', 'change user name', 'date', 'travel alert', 'directions', 'traffic', 'distance'] | right answer: distance
E [0.001 0.001 0.001 0.001 0.002 0.216 0.005 0.772]
G [0.001 0.001 0.001 0.005 0.002 0.216 0.001 0.772]
H [0.033 0.033 0.033 0.033 0.033 0.033 0.033 0.772]
W25 [0.001 0.001 0.001 0.001 0.002 0.162 0.004 0.829]
W75 [0.    0.    0.    0.    0.001 0.054 0.001 0.943]


## **[10] — Build a batch**

In [11]:
def make_batch(rows, run=None, rng=None):
    items = [build_input(r, rng) for r in rows]
    B = len(rows)
    L = max(len(it["input_ids"]) for it in items)       # longest sequence in this batch
    K = max(len(it["marker_pos"]) for it in items)      # most options in this batch
    input_ids   = torch.full((B, L), PAD, dtype=torch.long)
    attn        = torch.zeros((B, L), dtype=torch.long)
    marker_pos  = torch.zeros((B, K), dtype=torch.long) # empty slots point at position 0
    option_mask = torch.zeros((B, K), dtype=torch.bool)
    target      = torch.zeros((B, K), dtype=torch.float)
    perms = []
    for b, (row, it) in enumerate(zip(rows, items)):
        n_tok, k = len(it["input_ids"]), len(it["marker_pos"])
        input_ids[b, :n_tok] = torch.tensor(it["input_ids"])
        attn[b, :n_tok] = 1
        marker_pos[b, :k] = torch.tensor(it["marker_pos"])
        option_mask[b, :k] = True
        if run is not None:
            t = make_target(row, run)
            target[b, :k] = torch.tensor(t[it["perm"]], dtype=torch.float)   # into shown order
        perms.append(it["perm"])
    return {"input_ids": input_ids, "attention_mask": attn, "marker_pos": marker_pos,
            "option_mask": option_mask, "target": target, "perms": perms}

demo = make_batch(DATA["train"][:8], run="C", rng=random.Random(0))
for k, v in demo.items():
    if torch.is_tensor(v):
        print(f"{k:15s} shape {tuple(v.shape)}")
print("option_mask:\n", demo["option_mask"].int())

input_ids       shape (8, 102)
attention_mask  shape (8, 102)
marker_pos      shape (8, 8)
option_mask     shape (8, 8)
target          shape (8, 8)
option_mask:
 tensor([[1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0, 0, 0, 0]], dtype=torch.int32)


## **[11] — The student**

In [12]:
MODEL_INPUTS = ["input_ids", "attention_mask", "marker_pos", "option_mask"]

class Student(nn.Module):
    def __init__(self, name):
        super().__init__()
        try:
            self.body = AutoModel.from_pretrained(name, reference_compile=False)
        except (TypeError, ValueError):
            self.body = AutoModel.from_pretrained(name)
        self.body = self.body.float()
        self.scorer = nn.Linear(self.body.config.hidden_size, 1)        # 768 weights + 1 bias

    def forward(self, input_ids, attention_mask, marker_pos, option_mask):
        h = self.body(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (B, L, 768)
        idx = marker_pos.unsqueeze(-1).expand(-1, -1, h.size(-1))                           # (B, K, 768)
        markers = torch.gather(h, 1, idx)                                                   # (B, K, 768)
        scores = self.scorer(markers).squeeze(-1)                                           # (B, K)
        return scores.masked_fill(~option_mask, float("-inf"))                              # block empty slots

torch.manual_seed(SEED)
_student = Student(MODEL_NAME)
print(f"body:   {sum(p.numel() for p in _student.body.parameters()):,} numbers")
print(f"scorer: {sum(p.numel() for p in _student.scorer.parameters()):,} numbers")
with torch.no_grad():
    s = _student(**{k: demo[k] for k in MODEL_INPUTS})
print("scores shape:", tuple(s.shape))
print("untrained chances, row 0:", torch.softmax(s[0], dim=-1).numpy().round(3))
del _student

W1116 05:48:38.346000 782 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1116 05:48:38.388000 782 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


body:   149,014,272 numbers
scorer: 769 numbers
scores shape: (8, 8)
untrained chances, row 0: [0.234 0.185 0.17  0.151 0.133 0.128 0.    0.   ]


## **[12] — The loss**

In [13]:
def soft_cross_entropy(scores, target, option_mask):
    log_p = torch.log_softmax(scores.float(), dim=-1)
    log_p = log_p.masked_fill(~option_mask, 0.0)        # empty slots count as 0, not 0 × (−∞)
    return -(target * log_p).sum(dim=-1).mean()

# Check against Step 5's worked example
s = torch.tensor([[0.12, 2.35, 1.06, 0.06]])
m = torch.ones(1, 4, dtype=torch.bool)
for run, t in [("A", [0, 1, 0, 0]), ("B", [0.025, 0.925, 0.025, 0.025]), ("C", [0.08, 0.68, 0.18, 0.06])]:
    print(run, round(soft_cross_entropy(s, torch.tensor([t], dtype=torch.float), m).item(), 3))

A 0.395
B 0.54
C 0.943


## **[13] — Predicting and measuring**

In [14]:
@torch.no_grad()
def predict_scores(model, rows, bs=64):
    """Raw scores for every row, in stored order: one numpy array per row."""
    model.eval()
    out = []
    for i in range(0, len(rows), bs):
        batch = make_batch(rows[i:i + bs])                     # no rng: stored order
        feed = {k: batch[k].to(device) for k in MODEL_INPUTS}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_on()):
            s = model(**feed).float().cpu().numpy()
        for b, perm in enumerate(batch["perms"]):
            row_scores = np.empty(len(perm))
            row_scores[perm] = s[b, :len(perm)]                # shown order -> stored order
            out.append(row_scores)
    model.train()
    return out

def softmax_np(z):
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()

def per_row(probs_list, rows):
    """Confidence, right/wrong, and log-loss penalty for each question."""
    conf    = np.array([p.max() for p in probs_list])
    correct = np.array([float(p.argmax() == r["gold_index"]) for p, r in zip(probs_list, rows)])
    nll     = np.array([-np.log(max(p[r["gold_index"]], 1e-12)) for p, r in zip(probs_list, rows)])
    return conf, correct, nll

def ece_from(conf, correct, idx=None):
    if idx is not None:
        conf, correct = conf[idx], correct[idx]
    bins  = np.minimum((conf * N_BINS).astype(int), N_BINS - 1)
    said  = np.bincount(bins, weights=conf,    minlength=N_BINS)   # total confidence per group
    right = np.bincount(bins, weights=correct, minlength=N_BINS)   # total right answers per group
    return float(np.abs(right - said).sum() / len(conf))

def metrics(probs_list, rows):
    conf, correct, nll = per_row(probs_list, rows)
    return {"accuracy": correct.mean(), "ECE": ece_from(conf, correct), "log loss": nll.mean()}

## **[14] — One complete training run**

In [17]:
def lr_factor(step, total, warmup):
    """Warmup to 1.0, then a straight fall to 0 (Step 6e)."""
    if step < warmup:
        return (step + 1) / warmup
    return max(0.0, (total - step) / max(1, total - warmup))

def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def split_step(rows, micro):
    """Step 8: cut one training step into pieces of at most `micro` rows. Each piece's loss is weighted by its
    share of the step, so the pieces add up to exactly the whole step's loss, and therefore its slopes."""
    pieces = [rows[k:k + micro] for k in range(0, len(rows), micro)]
    return [(piece, len(piece) / len(rows)) for piece in pieces]

def add_lora(body):
    """Step 9: freeze the pre-trained body and add LoRA adapters to its attention and feed-forward layers."""
    from peft import LoraConfig, inject_adapter_in_model
    names = sorted({n.split(".")[-1] for n, m in body.named_modules() if isinstance(m, nn.Linear)})
    missing = [t for t in LORA_TARGETS if t not in names]
    assert not missing, f"no linear layers named {missing}; this model's linear layers are: {names}"
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     target_modules=LORA_TARGETS, bias="none")
    body = inject_adapter_in_model(cfg, body)
    for n, p in body.named_parameters():
        p.requires_grad = "lora_" in n                  # only the adapters learn
    trainable = sum(p.numel() for p in body.parameters() if p.requires_grad)
    total = sum(p.numel() for p in body.parameters())
    print(f"LoRA: {trainable:,} trainable numbers of {total:,} in the body ({trainable / total:.2%}); the rest is frozen")
    assert 0 < trainable < 0.05 * total, "LoRA should train a small share of the body"
    return body

def train_run(run):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = Student(MODEL_NAME)
    if ADAPT == "lora":
        model.body = add_lora(model.body)               # BEFORE the optimizer is built (your bug-table lesson)
    model = model.to(device)
    if GRAD_CKPT:
        try:
            model.body.gradient_checkpointing_enable()
            print("gradient checkpointing on: less memory, about a third slower")
        except Exception as e:
            print("gradient checkpointing not available for this model:", e)
    model.train()
    optimizer = torch.optim.AdamW([
        {"params": [p for p in model.body.parameters() if p.requires_grad],
         "lr": LR_LORA if ADAPT == "lora" else LR_BODY},        # full fine-tuning: every number is trainable
        {"params": model.scorer.parameters(), "lr": LR_HEAD},
    ], weight_decay=WEIGHT_DECAY)

    train = DATA["train"]
    steps_per_epoch = math.ceil(len(train) / BATCH_SIZE)
    total  = steps_per_epoch * EPOCHS
    warmup = max(1, int(WARMUP_FRAC * total))
    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: lr_factor(s, total, warmup))
    scaler = make_scaler(amp_on())
    print(f"run {run}: {len(train)} questions, {steps_per_epoch} steps per epoch, "
          f"{total} steps in total, warmup {warmup}")

    best, history, step = {"val_loss": float("inf")}, [], 0
    for epoch in range(EPOCHS):
        rng = random.Random(SEED * 1000 + epoch)           # same shuffle and swaps in every run
        order = list(range(len(train)))
        rng.shuffle(order)
        t0, losses = time.time(), []
        for i in range(0, len(order), BATCH_SIZE):
            step_rows, step_loss = [train[j] for j in order[i:i + BATCH_SIZE]], 0.0
            for piece, weight in split_step(step_rows, MICRO_BATCH):          # base: one piece of 16
                batch = make_batch(piece, run=run, rng=rng)
                feed = {k: batch[k].to(device) for k in MODEL_INPUTS}
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_on()):
                    scores = model(**feed)                                      # Steps 2-4: guess
                loss = soft_cross_entropy(scores, batch["target"].to(device), feed["option_mask"])  # Step 5
                if not torch.isfinite(loss):
                    raise RuntimeError("Loss became NaN or infinite. Set USE_AMP = False in Cell 4, "
                                       "rerun from Cell 4, and train again.")
                scaler.scale(loss * weight).backward()                          # slopes add up over pieces
                step_loss += loss.item() * weight
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)       # safety belt
            scaler.step(optimizer)                                              # nudge downhill
            scaler.update()
            schedule.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(step_loss)
            step += 1
            if step == min(20, steps_per_epoch) and device == "cuda":            # Step 8: speed and memory
                per_step = (time.time() - t0) / step
                print(f"  timing: {per_step:.2f}s per step, about {per_step * steps_per_epoch / 60:.0f} min per epoch "
                      f"| peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
            if step % 50 == 0:
                print(f"  epoch {epoch + 1} | step {step}/{total} | train loss {np.mean(losses[-50:]):.4f} "
                      f"| body lr {schedule.get_last_lr()[0]:.1e}")

        # Validation check: real answers, the same scoring for every run (Step 6g)
        val = metrics([softmax_np(s) for s in predict_scores(model, DATA["validation"])], DATA["validation"])
        history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses)),
                        "val_loss": float(val["log loss"]), "val_accuracy": float(val["accuracy"]),
                        "val_ECE": float(val["ECE"])})
        print(f"run {run} | epoch {epoch + 1} done in {time.time() - t0:.0f}s | train loss {np.mean(losses):.4f} "
              f"| val loss {val['log loss']:.4f} | val acc {val['accuracy']:.3f} | val ECE {val['ECE']:.3f}")
        if val["log loss"] < best["val_loss"]:
            best = {"val_loss": val["log loss"], "epoch": epoch + 1,
                    "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
            print(f"  -> new best student (epoch {epoch + 1})")

    if ADAPT == "lora":                                 # B starts at exactly zero: it must have moved
        moved = max(float(v.abs().max()) for k, v in model.state_dict().items() if "lora_B" in k)
        assert moved > 0, "LoRA B matrices are still all zero: the adapters never trained"
        print(f"adapters trained: largest |B| = {moved:.4f} (B starts at exactly zero)")
    model.load_state_dict(best["state"])
    print(f"run {run}: keeping epoch {best['epoch']}")
    preds = {"run": run, "best_epoch": best["epoch"], "history": history,
             "validation": predict_scores(model, DATA["validation"]),
             "test": predict_scores(model, DATA["test"]),
             "heldout": predict_scores(model, DATA["heldout"]),
             "snips": predict_scores(model, SNIPS) if "SNIPS" in globals() else None}   # Step 7
    with open(f"{OUT_DIR}/preds_{run}.pkl", "wb") as f:
        pickle.dump(preds, f)
    if SAVE_WEIGHTS:
        state = best["state"]
        if ADAPT == "lora":                             # adapters + scorer only: the frozen body is public anyway
            state = {k: v for k, v in state.items() if "lora_" in k or k.startswith("scorer")}
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in state.items()},
                   f"{OUT_DIR}/student_{run}.pt")                  # 16-bit
    del model, optimizer, best
    torch.cuda.empty_cache()
    return preds

## **[15] — Train the five students**

In [22]:
PREDS = {}
for run in TRAIN_RUNS:
    path = f"{OUT_DIR}/preds_{run}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            PREDS[run] = pickle.load(f)
        print(f"run {run}: already trained, loaded from Drive (kept epoch {PREDS[run]['best_epoch']})")
    else:
        print(f"\n========== training run {run} ==========")
        PREDS[run] = train_run(run)


========== training run W25 ==========


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


run W25: 16995 questions, 1063 steps per epoch, 3189 steps in total, warmup 318


/tmp/ipykernel_782/1441570461.py:87: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  schedule.step()


  timing: 0.29s per step, about 5 min per epoch | peak GPU memory 7.4 GB
  epoch 1 | step 50/3189 | train loss 1.1823 | body lr 4.8e-06
  epoch 1 | step 100/3189 | train loss 1.1927 | body lr 9.5e-06
  epoch 1 | step 150/3189 | train loss 1.0972 | body lr 1.4e-05
  epoch 1 | step 200/3189 | train loss 0.8572 | body lr 1.9e-05
  epoch 1 | step 250/3189 | train loss 0.6772 | body lr 2.4e-05
  epoch 1 | step 300/3189 | train loss 0.6203 | body lr 2.8e-05
  epoch 1 | step 350/3189 | train loss 0.5486 | body lr 3.0e-05
  epoch 1 | step 400/3189 | train loss 0.5209 | body lr 2.9e-05
  epoch 1 | step 450/3189 | train loss 0.4699 | body lr 2.9e-05
  epoch 1 | step 500/3189 | train loss 0.4647 | body lr 2.8e-05
  epoch 1 | step 550/3189 | train loss 0.5159 | body lr 2.8e-05
  epoch 1 | step 600/3189 | train loss 0.4983 | body lr 2.7e-05
  epoch 1 | step 650/3189 | train loss 0.4790 | body lr 2.7e-05
  epoch 1 | step 700/3189 | train loss 0.4781 | body lr 2.6e-05
  epoch 1 | step 750/3189 | trai

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


run W75: 16995 questions, 1063 steps per epoch, 3189 steps in total, warmup 318
  timing: 0.27s per step, about 5 min per epoch | peak GPU memory 7.4 GB
  epoch 1 | step 50/3189 | train loss 1.1806 | body lr 4.8e-06
  epoch 1 | step 100/3189 | train loss 1.1829 | body lr 9.5e-06
  epoch 1 | step 150/3189 | train loss 1.0686 | body lr 1.4e-05
  epoch 1 | step 200/3189 | train loss 0.8375 | body lr 1.9e-05
  epoch 1 | step 250/3189 | train loss 0.6333 | body lr 2.4e-05
  epoch 1 | step 300/3189 | train loss 0.5685 | body lr 2.8e-05
  epoch 1 | step 350/3189 | train loss 0.5058 | body lr 3.0e-05
  epoch 1 | step 400/3189 | train loss 0.4855 | body lr 2.9e-05
  epoch 1 | step 450/3189 | train loss 0.3779 | body lr 2.9e-05
  epoch 1 | step 500/3189 | train loss 0.3773 | body lr 2.8e-05
  epoch 1 | step 550/3189 | train loss 0.4371 | body lr 2.8e-05
  epoch 1 | step 600/3189 | train loss 0.4213 | body lr 2.7e-05
  epoch 1 | step 650/3189 | train loss 0.3808 | body lr 2.7e-05
  epoch 1 | step

## **[16] — Give every student a dial**

In [ ]:
GRID   = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))   # dial settings to try
B_GRID = np.linspace(-8.0, 8.0, 321)                             # nudge sizes to try (yes/no only)

def probs_with_dial(scores, qtype, T, b):
    z = np.array(scores, dtype=float) / T
    if qtype == "noul":
        z[1] += b
    return softmax_np(z)

def fit_dial(scores, rows):
    """The (dial, nudge) with the lowest validation log loss. Nudge only for yes/no."""
    if rows[0]["type"] == "noul":
        gap = np.array([s[1] - s[0] for s in scores])
        y   = np.array([r["gold_index"] for r in rows])
        with np.errstate(over="ignore"):
            z = gap[None, None, :] / GRID[:, None, None] + B_GRID[None, :, None]
            p = 1.0 / (1.0 + np.exp(-z))
        pen = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean(axis=2)
        i, j = np.unravel_index(np.argmin(pen), pen.shape)
        return float(GRID[i]), float(B_GRID[j])
    losses = [np.mean([-np.log(probs_with_dial(s, "choice", T, 0.0)[r["gold_index"]] + 1e-12)
                       for s, r in zip(scores, rows)]) for T in GRID]
    return float(GRID[int(np.argmin(losses))]), 0.0

def group_of(row):
    return (row["source"], row["type"])

def fit_dials(val_scores, val_rows):
    pooled = {}
    for qtype in ["choice", "noul"]:
        pairs = [(s, r) for s, r in zip(val_scores, val_rows) if r["type"] == qtype]
        pooled[qtype] = fit_dial(*zip(*pairs)) if len(pairs) >= MIN_FIT_ROWS else (1.0, 0.0)
    if DIAL_SETUP == "2 dials":
        return {}, pooled          # Step 1: one dial per question type, shared by all sources
    dials = {}
    for g in {group_of(r) for r in val_rows}:
        pairs = [(s, r) for s, r in zip(val_scores, val_rows) if group_of(r) == g]
        dials[g] = fit_dial(*zip(*pairs)) if len(pairs) >= MIN_FIT_ROWS else pooled[g[1]]
    return dials, pooled

def apply_dials(scores, rows, dials, pooled):
    return [probs_with_dial(s, r["type"], *dials.get(group_of(r), pooled[r["type"]]))
            for s, r in zip(scores, rows)]

TEST, HELD = DATA["test"], DATA["heldout"]
PROBS, PROBS_H = {}, {}             # name -> chances for every test / held-out question
for run in RUNS:
    dials, pooled = fit_dials(PREDS[run]["validation"], DATA["validation"])
    PROBS[f"Student {run}"]        = [softmax_np(s) for s in PREDS[run]["test"]]
    PROBS[f"Student {run} + dial"] = apply_dials(PREDS[run]["test"], TEST, dials, pooled)
    # Held-out's source never appears in validation, so apply_dials falls back to the POOLED dial:
    # fitted on validation questions from all six sources, before any PubMedQA answer is seen.
    PROBS_H[f"Student {run}"]        = [softmax_np(s) for s in PREDS[run]["heldout"]]
    PROBS_H[f"Student {run} + dial"] = apply_dials(PREDS[run]["heldout"], HELD, dials, pooled)
    print(f"Student {run} dials:", {f"{g[0].split('/')[-1]}/{g[1]}": (round(T, 2), round(b, 2))
                                    for g, (T, b) in sorted(dials.items())},
          f"| pooled choice dial (used for held-out): {pooled['choice'][0]:.2f}")
PROBS["Teacher"]   = [np.array(r["teacher_probs"]) / np.sum(r["teacher_probs"]) for r in TEST]
PROBS_H["Teacher"] = [np.array(r["teacher_probs"]) / np.sum(r["teacher_probs"]) for r in HELD]

## **[17] — The results tables**

In [ ]:
LABELS = {"A": "right answers only", "B": "softened evenly", "C": "soft_label",
          "D": f"{BLEND:.0%} human answer + soft_label", "E": "soft_label, teacher mistakes corrected",
          "F": f"{BLEND:.0%} human answer + corrected_soft_label"}

def results_table(P, rows, teacher_label):
    out = []
    for run in RUNS:
        raw  = metrics(P[f"Student {run}"], rows)
        dial = metrics(P[f"Student {run} + dial"], rows)
        out.append({"model": f"Student {run} ({LABELS[run]})", "kept epoch": PREDS[run]["best_epoch"],
                    "accuracy": raw["accuracy"], "ECE": raw["ECE"], "log loss": raw["log loss"],
                    "accuracy after dial": dial["accuracy"], "ECE after dial": dial["ECE"],
                    "log loss after dial": dial["log loss"]})
    t = metrics(P["Teacher"], rows)
    out.append({"model": teacher_label, "accuracy": t["accuracy"], "ECE": t["ECE"], "log loss": t["log loss"]})
    return pd.DataFrame(out).round(3)

print(f"TEST: the six sources the students trained on ({len(TEST)} questions)")
results = results_table(PROBS, TEST, "Teacher (dial + nudge)")
display(results)
results.to_csv(f"{OUT_DIR}/results_test.csv", index=False)

print(f"\nHELD-OUT PubMedQA: never trained on ({len(HELD)} questions; dials pooled from validation)")
results_h = results_table(PROBS_H, HELD, "Teacher (as-is, no dial)")
display(results_h)
results_h.to_csv(f"{OUT_DIR}/results_heldout.csv", index=False)

breakdown = []
for g in sorted({group_of(r) for r in TEST}):
    idx  = [i for i, r in enumerate(TEST) if group_of(r) == g]
    sub  = [TEST[i] for i in idx]
    line = {"source": g[0].split("/")[-1], "type": g[1], "n": len(idx)}
    for name, short in [(f"Student {r}", r) for r in RUNS] + [("Teacher", "teacher")]:
        m = metrics([PROBS[name][i] for i in idx], sub)
        line[f"acc {short}"] = m["accuracy"]
        line[f"ECE {short}"] = m["ECE"]
    breakdown.append(line)
breakdown = pd.DataFrame(breakdown).round(3)
display(breakdown)
breakdown.to_csv(f"{OUT_DIR}/results_by_source.csv", index=False)

## **[18] — Real difference or luck?**

In [ ]:
def make_boot(P, rows):
    """Bootstrap for one split: every model is scored on the SAME 1,000 redraws."""
    N = len(rows)
    idx_all = np.random.default_rng(SEED).integers(0, N, size=(N_BOOT, N))
    stats, cache = {name: per_row(P[name], rows) for name in P}, {}
    def boot(name, measure):
        if (name, measure) not in cache:
            conf, correct, nll = stats[name]
            if measure == "ECE":
                cache[name, measure] = np.array([ece_from(conf, correct, idx) for idx in idx_all])
            elif measure == "log loss":
                cache[name, measure] = np.array([nll[idx].mean() for idx in idx_all])
            else:
                cache[name, measure] = np.array([correct[idx].mean() for idx in idx_all])
        return cache[name, measure]
    return boot

# Every student gets its dial here: the dial is free, so "+ dial" is the fair comparison.
lines = []
for split_name, P, rows in [("test", PROBS, TEST), ("heldout", PROBS_H, HELD)]:
    boot = make_boot(P, rows)
    for cand, base in COMPARISONS:
        for measure, lower_is_better in [("accuracy", False), ("ECE", True), ("log loss", True)]:
            c = boot(f"Student {cand} + dial", measure)
            o = boot(f"Student {base} + dial", measure)
            diff = (o - c) if lower_is_better else (c - o)      # positive = candidate better
            lo, hi = np.percentile(diff, [2.5, 97.5])
            lines.append({"split": split_name, "comparison": f"{cand} vs {base}", "measure": measure,
                          "candidate better in": f"{int((diff > 0).sum())}/{N_BOOT}",
                          "average gap": round(float(diff.mean()), 4),
                          "95% range": f"[{lo:+.4f}, {hi:+.4f}]",
                          "verdict": f"{cand} better" if lo > 0 else
                                         (f"{cand} worse" if hi < 0 else "too close to call")})
boot_df = pd.DataFrame(lines)
display(boot_df)
boot_df.to_csv(f"{OUT_DIR}/bootstrap.csv", index=False)

## **[19] — Reliability diagrams**

In [ ]:
def reliability(ax, probs, rows, title):
    conf, correct, _ = per_row(probs, rows)
    bins = np.minimum((conf * N_BINS).astype(int), N_BINS - 1)
    for b in range(N_BINS):
        m = bins == b
        if m.any():
            center = (b + 0.5) / N_BINS
            ax.bar(center, correct[m].mean(), width=0.9 / N_BINS, color="#1D9E75", alpha=0.85)
            ax.text(center, correct[m].mean() + 0.02, str(int(m.sum())), ha="center", fontsize=7)
    ax.plot([0, 1], [0, 1], "--", color="gray")
    ax.set(xlim=(0, 1), ylim=(0, 1.08), xlabel="how sure it said it was", ylabel="how often it was right",
           title=f"{title}\nECE {ece_from(conf, correct):.3f}")

def reliability_grid(P, rows, title, path):
    panels = [f"Student {r}" for r in RUNS] + ["Teacher"]
    cols = 3 if len(panels) <= 6 else 4
    n_rows = math.ceil(len(panels) / cols)
    fig, axes = plt.subplots(n_rows, cols, figsize=(5 * cols, 4.5 * n_rows))
    for ax in axes.flat[len(panels):]:
        ax.axis("off")                               # hide unused panels
    for ax, name in zip(axes.flat, panels):
        reliability(ax, P[name], rows, name)
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()

reliability_grid(PROBS, TEST, "Test split (before dials)", f"{OUT_DIR}/reliability_test.png")
reliability_grid(PROBS_H, HELD, "Held-out PubMedQA (before dials)", f"{OUT_DIR}/reliability_heldout.png")

## **[20] — Training curves**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for run in RUNS:
    h = pd.DataFrame(PREDS[run]["history"])
    axes[0].plot(h["epoch"], h["train_loss"], marker="o", label=f"Run {run}")
    axes[1].plot(h["epoch"], h["val_loss"],   marker="o", label=f"Run {run}")
axes[0].set(title="training loss (each run's own target)", xlabel="epoch")
axes[1].set(title="validation loss (real answers)", xlabel="epoch")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.show()

## **[21] — (Optional): install Laya**

In [ ]:
!pip install -q laya

## **[22] — (Optional): Laya as an outside reference**

In [ ]:
from laya import Router
router = Router()

def laya_chances(row):
    if row["type"] == "noul":
        q = {"d": {"type": "noul", "instructions": row["instructions"]}}
    else:
        q = {"d": {"type": "choice", "instructions": row["instructions"],
                   "criteria": {o: o for o in row["options"]}}}
    ans = router.predict(row["state"], q)["answers"]["d"]
    if row["type"] == "noul":
        p_true = ans.get("probability", ans.get("value"))
        if p_true is None:
            raise KeyError(f"Couldn't find the yes/no probability in Laya's answer: {ans}")
        return np.array([1.0 - float(p_true), float(p_true)])
    p = np.array([float(ans["probabilities"].get(o, 0.0)) for o in row["options"]])
    if p.sum() == 0:
        raise KeyError(f"Laya's option names didn't match: {ans}")
    return p / p.sum()

t0 = time.time()
PROBS["Laya"] = [laya_chances(r) for r in TEST]
print(f"Laya scored {len(TEST)} test questions in {time.time() - t0:.0f}s")
print({k: round(float(v), 3) for k, v in metrics(PROBS["Laya"], TEST).items()})

## **[23] — Step 1: how many dials?**

In [ ]:
# Needs only: Cells 1-5, 13 and 15 (predictions loaded from Drive). No GPU, no retraining.
SCHEMES = ["no dial", "12 dials", "2 dials"]
REPEATS = 3                                   # random halvings of validation
GRID    = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))   # same search range as Cell 16
B_GRID  = np.linspace(-8.0, 8.0, 321)
VAL, TEST, HELD = DATA["validation"], DATA["test"], DATA["heldout"]

# 0. Saved predictions must line up with today's dataset rows
for run in RUNS:
    for split in ("validation", "test", "heldout"):
        assert len(PREDS[run][split]) == len(DATA[split]), f"{run}/{split}: row count differs"
        assert all(len(s) == len(r["options"]) for s, r in zip(PREDS[run][split], DATA[split]))
print("saved predictions line up with the dataset rows")

def group_of(row):
    return (row["source"], row["type"])

def fit_one(scores, rows):
    # One dial (and nudge for yes/no) with the lowest log loss on these rows
    y = np.array([r["gold_index"] for r in rows])
    if rows[0]["type"] == "noul":
        gap = np.array([s[1] - s[0] for s in scores])
        with np.errstate(over="ignore"):
            z = gap[None, None, :] / GRID[:, None, None] + B_GRID[None, :, None]
            p = 1.0 / (1.0 + np.exp(-z))
        pen = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean(axis=2)
        i, j = np.unravel_index(np.argmin(pen), pen.shape)
        return float(GRID[i]), float(B_GRID[j])
    total = np.zeros(len(GRID))                 # choice: all 200 dial values at once
    by_k = {}
    for s, g in zip(scores, y):
        by_k.setdefault(len(s), []).append((s, g))
    for items in by_k.values():                 # questions with the same number of options together
        S = np.array([s for s, _ in items], dtype=float)
        g = np.array([g for _, g in items])
        Z = S[None] / GRID[:, None, None]
        Z = Z - Z.max(axis=2, keepdims=True)
        logp = Z - np.log(np.exp(Z).sum(axis=2, keepdims=True))
        total += -logp[:, np.arange(len(g)), g].sum(axis=1)
    return float(GRID[int(np.argmin(total))]), 0.0

def fit_scheme(scores, rows, scheme):
    # Returns (per-group dials, per-type dials). Groups without their own dial use the per-type one.
    if scheme == "no dial":
        return {}, {"choice": (1.0, 0.0), "noul": (1.0, 0.0)}
    pooled = {}
    for qtype in ("choice", "noul"):
        idx = [i for i, r in enumerate(rows) if r["type"] == qtype]
        pooled[qtype] = (fit_one([scores[i] for i in idx], [rows[i] for i in idx])
                         if len(idx) >= MIN_FIT_ROWS else (1.0, 0.0))
    if scheme == "2 dials":
        return {}, pooled
    dials = {}
    for g in {group_of(r) for r in rows}:
        idx = [i for i, r in enumerate(rows) if group_of(r) == g]
        if len(idx) >= MIN_FIT_ROWS:
            dials[g] = fit_one([scores[i] for i in idx], [rows[i] for i in idx])
    return dials, pooled

def apply_scheme(scores, rows, fitted):
    dials, pooled = fitted
    out = []
    for s, r in zip(scores, rows):
        T, b = dials.get(group_of(r), pooled[r["type"]])
        z = np.array(s, dtype=float) / T
        if r["type"] == "noul":
            z[1] += b
        out.append(softmax_np(z))
    return out

# 1. THE DECISION: cross-fitting on validation (fit on one half, score on the other)
cv = []
for run in RUNS:
    sv = PREDS[run]["validation"]
    for scheme in SCHEMES:
        lls, eces = [], []
        for rep in range(REPEATS):
            order = np.random.default_rng(SEED + rep).permutation(len(VAL))
            h1, h2 = order[: len(VAL) // 2], order[len(VAL) // 2:]
            for fit_idx, score_idx in ((h1, h2), (h2, h1)):
                fitted = fit_scheme([sv[i] for i in fit_idx], [VAL[i] for i in fit_idx], scheme)
                rows = [VAL[i] for i in score_idx]
                m = metrics(apply_scheme([sv[i] for i in score_idx], rows, fitted), rows)
                lls.append(m["log loss"])
                eces.append(m["ECE"])
        cv.append({"student": run, "dials": scheme, "log loss": np.mean(lls), "ECE": np.mean(eces)})
cv = pd.DataFrame(cv)
print("\nDECISION (validation, cross-fitted): lower is better")
display(cv.pivot(index="student", columns="dials", values="log loss")[SCHEMES].round(4)
          .rename(columns=lambda c: f"log loss: {c}"))
display(cv.pivot(index="student", columns="dials", values="ECE")[SCHEMES].round(4)
          .rename(columns=lambda c: f"ECE: {c}"))
avg = cv.groupby("dials")["log loss"].mean()[SCHEMES]
CHOSEN = avg.idxmin()
print("average validation log loss over all five students:", {k: round(float(v), 4) for k, v in avg.items()})
print("best setup per student:", cv.loc[cv.groupby("student")["log loss"].idxmin()].set_index("student")["dials"].to_dict())
print(f"\n=> chosen for every later step: {CHOSEN}")

# 2. THE REPORT: each setup fitted on ALL of validation, scored once on test and held-out
rep_rows = []
for run in RUNS:
    for scheme in SCHEMES:
        fitted = fit_scheme(PREDS[run]["validation"], VAL, scheme)
        mt = metrics(apply_scheme(PREDS[run]["test"], TEST, fitted), TEST)
        mh = metrics(apply_scheme(PREDS[run]["heldout"], HELD, fitted), HELD)
        rep_rows.append({"student": run, "dials": scheme,
                         "test acc": mt["accuracy"], "test ECE": mt["ECE"], "test log loss": mt["log loss"],
                         "held-out acc": mh["accuracy"], "held-out ECE": mh["ECE"], "held-out log loss": mh["log loss"]})
report = pd.DataFrame(rep_rows).round(3)
print("\nREPORT (test and held-out): for information, not for choosing")
display(report)

# 3. Correctness check: the 12-dial setup must reproduce Cell 17's "after dial" test numbers
CELL17 = {"A": (0.908, 0.018, 0.268), "B": (0.917, 0.025, 0.322), "C": (0.900, 0.019, 0.340),
          "D": (0.922, 0.015, 0.262), "E": (0.919, 0.016, 0.271)}
for run, expected in CELL17.items():
    got = tuple(round(float(x), 3) for x in
                report[(report["student"] == run) & (report["dials"] == "12 dials")][["test acc", "test ECE", "test log loss"]].values[0])
    ok = np.allclose(got, expected, atol=0.0015)
    print(f"12-dial check, Student {run}: {got} vs Cell 17 {expected} -> {'MATCH' if ok else 'DIFFERENT'}")

cv.to_csv(f"{OUT_DIR}/step1_dials_validation.csv", index=False)
report.to_csv(f"{OUT_DIR}/step1_dials_report.csv", index=False)

## **[24] — Step-2: Results across seeds**

In [23]:
# Needs only: Cells 1-5 and 13. Loads every seed's predictions from Drive. No GPU.
def seed_dir(seed):
    return BASE_DIR if seed == 0 else f"{BASE_DIR}/seed{seed}"

ALL_PREDS = {}
for seed in SEEDS_ALL:
    for run in RUNS:
        path = f"{seed_dir(seed)}/preds_{run}.pkl"
        if os.path.exists(path):
            with open(path, "rb") as f:
                ALL_PREDS[seed, run] = pickle.load(f)
        else:
            print("not found yet:", path)
SEEDS_DONE = sorted({s for s, _ in ALL_PREDS})
assert all((s, r) in ALL_PREDS for s in SEEDS_DONE for r in RUNS), "each seed needs all five runs finished"
for (seed, run), P in ALL_PREDS.items():       # every saved prediction must line up with today's rows
    for split in ("validation", "test", "heldout"):
        assert len(P[split]) == len(DATA[split]), f"seed {seed}, run {run}, {split}: row count differs"
print("seeds found:", SEEDS_DONE)

# The 2-dial setup from Step 1 (same fitting as Cell 23)
GRID   = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))
B_GRID = np.linspace(-8.0, 8.0, 321)

def fit_one(scores, rows):
    y = np.array([r["gold_index"] for r in rows])
    if rows[0]["type"] == "noul":
        gap = np.array([s[1] - s[0] for s in scores])
        with np.errstate(over="ignore"):
            z = gap[None, None, :] / GRID[:, None, None] + B_GRID[None, :, None]
            p = 1.0 / (1.0 + np.exp(-z))
        pen = -(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean(axis=2)
        i, j = np.unravel_index(np.argmin(pen), pen.shape)
        return float(GRID[i]), float(B_GRID[j])
    total, by_k = np.zeros(len(GRID)), {}
    for s, g in zip(scores, y):
        by_k.setdefault(len(s), []).append((s, g))
    for items in by_k.values():
        S = np.array([s for s, _ in items], dtype=float)
        g = np.array([g for _, g in items])
        Z = S[None] / GRID[:, None, None]
        Z = Z - Z.max(axis=2, keepdims=True)
        logp = Z - np.log(np.exp(Z).sum(axis=2, keepdims=True))
        total += -logp[:, np.arange(len(g)), g].sum(axis=1)
    return float(GRID[int(np.argmin(total))]), 0.0

def two_dials(scores, rows):
    pooled = {}
    for qtype in ("choice", "noul"):
        idx = [i for i, r in enumerate(rows) if r["type"] == qtype]
        pooled[qtype] = (fit_one([scores[i] for i in idx], [rows[i] for i in idx])
                         if len(idx) >= MIN_FIT_ROWS else (1.0, 0.0))
    return pooled

def apply_two(scores, rows, pooled):
    out = []
    for s, r in zip(scores, rows):
        T, b = pooled[r["type"]]
        z = np.array(s, dtype=float) / T
        if r["type"] == "noul":
            z[1] += b
        out.append(softmax_np(z))
    return out

# Every student: fit its 2 dials on validation, score test and held-out
rows_out = []
for (seed, run), P in sorted(ALL_PREDS.items()):
    pooled = two_dials(P["validation"], DATA["validation"])
    for split in ("test", "heldout"):
        m = metrics(apply_two(P[split], DATA[split], pooled), DATA[split])
        rows_out.append({"seed": seed, "run": run, "split": split,
                         **{k: float(v) for k, v in m.items()}})
PER_SEED = pd.DataFrame(rows_out)
MEASURES = ["accuracy", "ECE", "log loss"]

# 1. Mean and range across seeds
def mean_range(x):
    return f"{x.mean():.3f}  ({x.min():.3f} to {x.max():.3f})"
summary = (PER_SEED.groupby(["split", "run"])
           .agg(**{m: (m, mean_range) for m in MEASURES}).reset_index())
print(f"\n1. MEAN (lowest to highest) across {len(SEEDS_DONE)} seeds, 2 dials")
display(summary)

# 2. How big is training luck? Each run's spread (highest minus lowest) across seeds
spread = PER_SEED.groupby(["split", "run"])[MEASURES].agg(lambda x: x.max() - x.min()).round(4).reset_index()
print("\n2. TRAINING LUCK: how much each run moved between seeds (highest minus lowest)")
display(spread)

# 3. Does each comparison hold in every seed?
lines = []
for split in ("test", "heldout"):
    for cand, base in COMPARISONS:
        for measure, lower_is_better in [("accuracy", False), ("ECE", True), ("log loss", True)]:
            gaps = []
            for seed in SEEDS_DONE:
                pick = lambda r: PER_SEED[(PER_SEED.seed == seed) & (PER_SEED.run == r) &
                                          (PER_SEED.split == split)][measure].iloc[0]
                c, o = pick(cand), pick(base)
                gaps.append((o - c) if lower_is_better else (c - o))   # positive = candidate better
            gaps = np.array(gaps)
            wins = int((gaps > 0).sum())
            lines.append({"split": split, "comparison": f"{cand} vs {base}", "measure": measure,
                          "seeds where candidate better": f"{wins}/{len(gaps)}",
                          "average gap": round(float(gaps.mean()), 4),
                          "gap range": f"{gaps.min():+.4f} to {gaps.max():+.4f}",
                          "verdict": (f"{cand} better in every seed" if wins == len(gaps) else
                                      f"{cand} worse in every seed" if wins == 0 else "depends on the seed")})
CONSISTENCY = pd.DataFrame(lines)
print("\n3. CONSISTENCY: does each comparison hold in every seed? (positive gap = candidate better)")
display(CONSISTENCY)

# Which epoch each student kept
epochs = pd.DataFrame([{"seed": s, "run": r, "kept epoch": P["best_epoch"]} for (s, r), P in ALL_PREDS.items()])
print("\nkept epoch per student:")
display(epochs.pivot(index="seed", columns="run", values="kept epoch"))

PER_SEED.to_csv(f"{BASE_DIR}/step2_per_seed.csv", index=False)
CONSISTENCY.to_csv(f"{BASE_DIR}/step2_consistency.csv", index=False)

seeds found: [0, 1, 2]

1. MEAN (lowest to highest) across 3 seeds, 2 dials


,split,run,accuracy,ECE,log loss
0,heldout,A,0.580 (0.538 to 0.614),0.130 (0.097 to 0.171),0.970 (0.945 to 0.997)
1,heldout,B,0.579 (0.568 to 0.585),0.092 (0.020 to 0.149),0.987 (0.957 to 1.005)
2,heldout,C,0.583 (0.576 to 0.592),0.055 (0.027 to 0.106),0.940 (0.926 to 0.952)
3,heldout,D,0.574 (0.548 to 0.595),0.044 (0.028 to 0.064),0.947 (0.878 to 1.017)
4,heldout,E,0.596 (0.591 to 0.605),0.086 (0.048 to 0.107),0.935 (0.924 to 0.946)
5,heldout,F,0.595 (0.584 to 0.607),0.046 (0.019 to 0.068),0.914 (0.888 to 0.953)
6,test,A,0.912 (0.909 to 0.917),0.011 (0.006 to 0.019),0.247 (0.230 to 0.267)
7,test,B,0.918 (0.913 to 0.923),0.022 (0.016 to 0.026),0.281 (0.268 to 0.289)
8,test,C,0.892 (0.886 to 0.898),0.021 (0.020 to 0.023),0.310 (0.307 to 0.313)
9,test,D,0.915 (0.909 to 0.918),0.016 (0.015 to 0.019),0.248 (0.241 to 0.256)



2. TRAINING LUCK: how much each run moved between seeds (highest minus lowest)


,split,run,accuracy,ECE,log loss
0,heldout,A,0.0760,0.0738,0.0522
1,heldout,B,0.0170,0.1289,0.0476
2,heldout,C,0.0160,0.0790,0.0257
3,heldout,D,0.0470,0.0356,0.1386
4,heldout,E,0.0140,0.0586,0.0221
5,heldout,F,0.0230,0.0484,0.0652
6,test,A,0.0078,0.0132,0.0372
7,test,B,0.0100,0.0101,0.0208
8,test,C,0.0117,0.0031,0.0058
9,test,D,0.0083,0.0040,0.0153



3. CONSISTENCY: does each comparison hold in every seed? (positive gap = candidate better)


,split,comparison,measure,seeds where candidate better,average gap,gap range,verdict
0,test,C vs A,accuracy,0/3,-0.0195,-0.0250 to -0.0106,C worse in every seed
1,test,C vs A,ECE,0/3,-0.0100,-0.0177 to -0.0015,C worse in every seed
2,test,C vs A,log loss,0/3,-0.0635,-0.0806 to -0.0393,C worse in every seed
3,test,C vs B,accuracy,0/3,-0.0260,-0.0317 to -0.0195,C worse in every seed
4,test,C vs B,ECE,2/3,0.0010,-0.0073 to +0.0060,depends on the seed
...,...,...,...,...,...,...,...
73,heldout,E vs D,ECE,0/3,-0.0414,-0.0613 to -0.0200,E worse in every seed
74,heldout,E vs D,log loss,2/3,0.0115,-0.0460 to +0.0705,depends on the seed
75,heldout,E vs C,accuracy,2/3,0.0137,-0.0010 to +0.0250,depends on the seed
76,heldout,E vs C,ECE,1/3,-0.0307,-0.0798 to +0.0578,depends on the seed



kept epoch per student:


run,A,B,C,D,E,F
seed,,,,,,
0,1,3,2,2,2,2
1,2,3,2,2,2,3
2,2,3,3,3,3,3


## **[25] — Step 3: Ensembles**

In [24]:
# Needs: Cells 1-5, 13 and 24 (which loads every seed and defines the 2-dial helpers). No GPU.
assert SEEDS_DONE == SEEDS_ALL, "train all three seeds first (Step 2)"

# Each single student, after its own 2 dials (fitted on validation)
SINGLE, RAW = {}, {}
for (seed, run), P in ALL_PREDS.items():
    pooled = two_dials(P["validation"], DATA["validation"])
    for split in ("test", "heldout"):
        SINGLE[seed, run, split] = apply_two(P[split], DATA[split], pooled)
        RAW[seed, run, split] = [softmax_np(np.array(s, dtype=float)) for s in P[split]]

def average(students):
    # Average several students' chances, question by question, option by option
    return [np.mean([st[i] for st in students], axis=0) for i in range(len(students[0]))]

# The official ensemble (decided before looking): average of the dialed students
ENS, ENS_RAW = {}, {}
for run in RUNS:
    for split in ("test", "heldout"):
        ENS[run, split] = average([SINGLE[s, run, split] for s in SEEDS_ALL])
        ENS_RAW[run, split] = average([RAW[s, run, split] for s in SEEDS_ALL])   # for information only

# 1. One student vs the ensemble
rows = []
for split in ("test", "heldout"):
    for run in RUNS:
        singles = [metrics(SINGLE[s, run, split], DATA[split]) for s in SEEDS_ALL]
        ens = metrics(ENS[run, split], DATA[split])
        raw = metrics(ENS_RAW[run, split], DATA[split])
        rows.append({"split": split, "run": run,
                     "accuracy: one student": np.mean([m["accuracy"] for m in singles]),
                     "accuracy: ensemble": ens["accuracy"],
                     "ECE: one student": np.mean([m["ECE"] for m in singles]),
                     "ECE: ensemble": ens["ECE"],
                     "log loss: one student": np.mean([m["log loss"] for m in singles]),
                     "log loss: best single": min(m["log loss"] for m in singles),
                     "log loss: ensemble": ens["log loss"],
                     "log loss: ensemble without dials": raw["log loss"]})
ENS_TABLE = pd.DataFrame(rows)
print("1. ONE STUDENT (average of 3 seeds) vs THE ENSEMBLE OF ALL 3")
display(ENS_TABLE.round(3))

# 2. The guarantee: an ensemble's log loss is never worse than its members' average
for r in rows:
    assert r["log loss: ensemble"] <= r["log loss: one student"] + 1e-12, f"{r['split']} {r['run']}: Jensen check failed"
print("\n2. Jensen check passed: every ensemble's log loss <= its members' average")

# 3. Ensemble vs ensemble, paired bootstrap (the seeds are averaged in; question luck remains)
def boot_values(P, rows_, measure, idx_all):
    conf, correct, nll = per_row(P, rows_)
    if measure == "ECE":
        return np.array([ece_from(conf, correct, idx) for idx in idx_all])
    if measure == "log loss":
        return np.array([nll[idx].mean() for idx in idx_all])
    return np.array([correct[idx].mean() for idx in idx_all])

lines = []
for split in ("test", "heldout"):
    rows_ = DATA[split]
    idx_all = np.random.default_rng(SEED).integers(0, len(rows_), size=(N_BOOT, len(rows_)))
    cache = {}
    def bv(run, measure):
        if (run, measure) not in cache:
            cache[run, measure] = boot_values(ENS[run, split], rows_, measure, idx_all)
        return cache[run, measure]
    for cand, base in COMPARISONS:
        for measure, lower_is_better in [("accuracy", False), ("ECE", True), ("log loss", True)]:
            c, o = bv(cand, measure), bv(base, measure)
            diff = (o - c) if lower_is_better else (c - o)          # positive = candidate better
            lo, hi = np.percentile(diff, [2.5, 97.5])
            wins, ties = int((diff > 0).sum()), int((diff == 0).sum())
            lines.append({"split": split, "comparison": f"{cand} vs {base} (ensembles)", "measure": measure,
                          "candidate better in": f"{wins}/{N_BOOT}" + (f" ({ties} ties)" if ties else ""),
                          "average gap": round(float(diff.mean()), 4),
                          "95% range": f"[{lo:+.4f}, {hi:+.4f}]",
                          "verdict": f"{cand} better" if lo > 0 else
                                     (f"{cand} worse" if hi < 0 else "too close to call")})
ENS_BOOT = pd.DataFrame(lines)
print("\n3. ENSEMBLE vs ENSEMBLE (paired bootstrap, 2 dials per member)")
display(ENS_BOOT)

# 4. Held-out baselines, for context
freq = pd.Series([r["gold"] for r in DATA["heldout"]]).value_counts(normalize=True)
print(f"\n4. Held-out baselines: always answering '{freq.index[0]}' scores accuracy {freq.iloc[0]:.3f}; "
      f"knowing only how often each answer occurs gives log loss {float(-(freq * np.log(freq)).sum()):.3f}")

ENS_TABLE.to_csv(f"{BASE_DIR}/step3_ensembles.csv", index=False)
ENS_BOOT.to_csv(f"{BASE_DIR}/step3_ensemble_bootstrap.csv", index=False)

1. ONE STUDENT (average of 3 seeds) vs THE ENSEMBLE OF ALL 3


,split,run,accuracy: one student,accuracy: ensemble,ECE: one student,ECE: ensemble,log loss: one student,log loss: best single,log loss: ensemble,log loss: ensemble without dials
0,test,A,0.912,0.922,0.011,0.018,0.247,0.230,0.221,0.229
1,test,B,0.918,0.925,0.022,0.009,0.281,0.268,0.249,0.259
2,test,C,0.892,0.895,0.021,0.013,0.310,0.307,0.292,0.311
3,test,D,0.915,0.920,0.016,0.008,0.248,0.241,0.231,0.241
4,test,E,0.912,0.918,0.015,0.013,0.261,0.252,0.241,0.277
5,test,F,0.919,0.928,0.015,0.006,0.245,0.241,0.225,0.234
6,heldout,A,0.580,0.595,0.130,0.153,0.970,0.945,0.964,0.924
7,heldout,B,0.579,0.582,0.092,0.052,0.987,0.957,0.948,0.939
8,heldout,C,0.583,0.595,0.055,0.065,0.940,0.926,0.932,0.951
9,heldout,D,0.574,0.589,0.044,0.054,0.947,0.878,0.928,0.942



2. Jensen check passed: every ensemble's log loss <= its members' average

3. ENSEMBLE vs ENSEMBLE (paired bootstrap, 2 dials per member)


,split,comparison,measure,candidate better in,average gap,95% range,verdict
0,test,C vs A (ensembles),accuracy,0/1000,-0.0266,"[-0.0390, -0.0139]",C worse
1,test,C vs A (ensembles),ECE,715/1000,0.0038,"[-0.0114, +0.0170]",too close to call
2,test,C vs A (ensembles),log loss,0/1000,-0.0714,"[-0.0922, -0.0520]",C worse
3,test,C vs B (ensembles),accuracy,0/1000,-0.0302,"[-0.0428, -0.0167]",C worse
4,test,C vs B (ensembles),ECE,199/1000,-0.0052,"[-0.0175, +0.0075]",too close to call
...,...,...,...,...,...,...,...
73,heldout,E vs D (ensembles),ECE,1/1000,-0.0284,"[-0.0452, -0.0097]",E worse
74,heldout,E vs D (ensembles),log loss,623/1000,0.0015,"[-0.0065, +0.0104]",too close to call
75,heldout,E vs C (ensembles),accuracy,172/1000 (23 ties),-0.0083,"[-0.0270, +0.0100]",too close to call
76,heldout,E vs C (ensembles),ECE,18/1000,-0.0196,"[-0.0384, -0.0010]",E worse



4. Held-out baselines: always answering 'yes' scores accuracy 0.552; knowing only how often each answer occurs gives log loss 0.937


## **[26] — SNIPS: build the questions**
Run before training in Step 7. Fixed seed: every student sees identical questions.

In [16]:
# Step 6/7: BUILD the SNIPS questions only. Needs: Cells 1-7.
# Run it BEFORE training (Cells 14-15) in Step 7, so each new student also answers SNIPS.
import zipfile, glob, re, urllib.request
from collections import Counter
SNIPS_URL        = "https://github.com/sonos/nlu-benchmark/archive/refs/heads/master.zip"   # authors' repo, CC0-1.0
SNIPS_DIR        = "/content/snips"
SNIPS_PER_INTENT = 150
SNIPS_DESC = {   # written by us, for the student, like the v2 descriptions
    "add to playlist":        "add a song or artist to a playlist",
    "book restaurant":        "reserve a table at a restaurant",
    "get weather":            "ask about the weather or the forecast",
    "play music":             "play a song, album, artist, or playlist",
    "rate book":              "give a rating to a book or other written work",
    "search creative work":   "find a book, song, film, show, or game",
    "search screening event": "find showtimes, or where a film is playing",
}
SNIPS_SOURCE = "sonos/nlu-benchmark (SNIPS 2017)"
SNIPS_SEED   = 0          # fixed on purpose: NOT the training SEED, so every student sees identical questions

# 1. Download and read the authors' files
os.makedirs(SNIPS_DIR, exist_ok=True)
zpath = f"{SNIPS_DIR}/nlu-benchmark.zip"
if not os.path.exists(zpath):
    urllib.request.urlretrieve(SNIPS_URL, zpath)
with zipfile.ZipFile(zpath) as z:
    z.extractall(SNIPS_DIR)

def read_json_any(path):
    raw = open(path, "rb").read()
    for enc in ("utf-8", "latin-1"):           # a few of these files aren't valid UTF-8
        try:
            return json.loads(raw.decode(enc))
        except (UnicodeDecodeError, json.JSONDecodeError):
            pass
    raise ValueError(f"can't read {path}")

def humanize_intent(name):                     # "SearchScreeningEvent" -> "search screening event"
    return " ".join(w.lower() for w in re.findall(r"[A-Z][a-z]*", name))

def utterances(path):
    d = read_json_any(path)
    (intent, items), = d.items()
    return humanize_intent(intent), [" ".join("".join(seg["text"] for seg in it["data"]).split()) for it in items]

base = glob.glob(f"{SNIPS_DIR}/**/2017-06-custom-intent-engines", recursive=True)
assert len(base) == 1, f"couldn't find the 2017 folder: {base}"
POOL, SEEN = {}, {}
for kind in ("validate_*.json", "train_*_full.json"):       # validation files first, then the training files
    for path in sorted(glob.glob(f"{base[0]}/*/{kind}")):
        intent, texts = utterances(path)
        POOL.setdefault(intent, []); SEEN.setdefault(intent, set())
        for t in texts:
            if t and t.lower() not in SEEN[intent]:
                POOL[intent].append(t); SEEN[intent].add(t.lower())
SNIPS_NAMES = sorted(POOL)
print("intents:", SNIPS_NAMES)
assert set(SNIPS_NAMES) == set(SNIPS_DESC), f"unexpected intents: {SNIPS_NAMES}"
assert all(len(v) >= SNIPS_PER_INTENT for v in POOL.values()), {k: len(v) for k, v in POOL.items()}

# 2. Overlap check: is any SNIPS request word-for-word in the training data?
train_texts = {r["state"].strip().lower() for r in DATA["train"]}
picked = {k: v[:SNIPS_PER_INTENT] for k, v in POOL.items()}
overlap = sum(t.lower() in train_texts for v in picked.values() for t in v)
print(f"SNIPS requests found word-for-word in the training data: {overlap} of {sum(map(len, picked.values()))}")

# 3. Questions, built like the v2 questions (descriptions, 3-7 options, 30% yes/no), seeded
rng = random.Random(f"{SNIPS_SEED}-snips")
SNIPS = []
for gi, intent in enumerate(SNIPS_NAMES):
    for j, text in enumerate(picked[intent]):
        others = [k for k in range(len(SNIPS_NAMES)) if k != gi]
        if rng.random() < 0.3:
            is_true = rng.random() < 0.5
            cand = gi if is_true else rng.choice(others)
            label = SNIPS_NAMES[cand]
            q = dict(type="noul", options=["false", "true"], option_descriptions=["", ""], candidate=label,
                     instructions=f"Is this request to a voice assistant asking for: {label} ({SNIPS_DESC[label]})?",
                     gold_index=int(is_true))
        else:
            k = rng.randint(3, len(SNIPS_NAMES))
            opts = [gi] + rng.sample(others, k - 1)
            rng.shuffle(opts)
            names = [SNIPS_NAMES[o] for o in opts]
            q = dict(type="choice", options=names, option_descriptions=[SNIPS_DESC[n] for n in names], candidate="",
                     instructions="Which intent best describes this request to a voice assistant?",
                     gold_index=opts.index(gi))
        SNIPS.append(dict(id=f"snips-{gi}-{j:04d}", source=SNIPS_SOURCE, state=text,
                          gold=q["options"][q["gold_index"]], **q))
pretokenize(SNIPS)
print(f"{len(SNIPS)} questions:", dict(Counter(r["type"] for r in SNIPS)))
import hashlib
SNIPS_FINGERPRINT = hashlib.md5(json.dumps([(r["id"], r["options"], r["gold_index"], r["state"]) for r in SNIPS])
                                .encode()).hexdigest()[:12]
print("question-set fingerprint:", SNIPS_FINGERPRINT, "(must be the same in every session)")

intents: ['add to playlist', 'book restaurant', 'get weather', 'play music', 'rate book', 'search creative work', 'search screening event']
SNIPS requests found word-for-word in the training data: 0 of 1050
1050 questions: {'choice': 734, 'noul': 316}
question-set fingerprint: b0096e8b4312 (must be the same in every session)


## **[27] — SNIPS: seed 0's students answer**
Reuses earlier answers if they were for these exact questions; otherwise re-scores.

In [25]:
# Step 7: seed 0's students answer the SNIPS questions. Needs: Cells 1-11, 13, 24 and 26. GPU recommended.
SEED0_FILE = f"{BASE_DIR}/step6_snips_scores_seed0.pkl"
def signature(rows):
    return [(r["id"], r["type"], tuple(r["options"]), r["gold_index"], r["state"]) for r in rows]

SNIPS_SCORES = None
if os.path.exists(SEED0_FILE):
    with open(SEED0_FILE, "rb") as f:
        saved = pickle.load(f)
    if signature(saved["rows"]) == signature(SNIPS):
        SNIPS_SCORES = saved["scores"]
        print("reusing seed 0's earlier SNIPS answers: they were for exactly these questions")
    else:
        print("seed 0's earlier SNIPS answers were for different questions (the old seeding): re-scoring")

if SNIPS_SCORES is None:
    CHECK_N, SNIPS_SCORES = 64, {}
    for run in RUNS:
        wpath = f"{BASE_DIR}/student_{run}.pt"
        assert os.path.exists(wpath), f"no saved weights for run {run} (seed 0): {wpath}"
        model = Student(MODEL_NAME)
        model.load_state_dict(torch.load(wpath, map_location="cpu"))
        model = model.to(device)
        again = predict_scores(model, DATA["test"][:CHECK_N])
        saved_test = ALL_PREDS[0, run]["test"][:CHECK_N]
        worst = max(float(np.max(np.abs(np.asarray(a) - np.asarray(b)))) for a, b in zip(again, saved_test))
        same_pick = all(int(np.argmax(a)) == int(np.argmax(b)) for a, b in zip(again, saved_test))
        assert worst < 0.05 and same_pick, f"run {run}: weights don't reproduce the saved predictions (max gap {worst:.4f})"
        SNIPS_SCORES[run] = predict_scores(model, SNIPS)
        print(f"run {run}: weights reproduce its saved test predictions (max gap {worst:.4f}); SNIPS answered")
        del model
        torch.cuda.empty_cache()
    with open(SEED0_FILE, "wb") as f:
        pickle.dump({"rows": [{k: r[k] for k in ("id", "type", "options", "gold_index", "state")} for r in SNIPS],
                     "scores": SNIPS_SCORES}, f)
print("seed 0: SNIPS answers ready for", sorted(SNIPS_SCORES))

reusing seed 0's earlier SNIPS answers: they were for exactly these questions
seed 0: SNIPS answers ready for ['A', 'B', 'C', 'D', 'E', 'F']


## **[28] — Step 7: SNIPS across all three seeds**
Sanity check of the reruns, then seeds, ensembles, and both checks.

In [26]:
# Step 7 analysis. Needs: Cells 1-5, 13, 24, 26 and 27, and both reruns (seed 1 and seed 2) finished.
def rerun_dir(seed):
    return f"{BASE_DIR}/seed{seed}_rerun"

RER = {}
for seed in (1, 2):
    for run in RUNS:
        path = f"{rerun_dir(seed)}/preds_{run}.pkl"
        assert os.path.exists(path), f"rerun not finished: {path}"
        with open(path, "rb") as f:
            P = pickle.load(f)
        assert P.get("snips") is not None and len(P["snips"]) == len(SNIPS), \
            f"seed {seed}, run {run}: no SNIPS answers saved (was Cell 26 run before training?)"
        RER[seed, run] = P
print("reruns found: seeds 1 and 2, all", len(RUNS), "runs")

# 1. Sanity: each retrained student vs the original student of the same seed, test split, 2 dials
spread = (PER_SEED[PER_SEED.split == "test"].groupby("run")[["accuracy", "log loss"]]
          .agg(lambda x: x.max() - x.min()))                          # Step 2's normal seed-to-seed spread
san = []
for (seed, run), P in sorted(RER.items()):
    O = ALL_PREDS[seed, run]
    mo = metrics(apply_two(O["test"], DATA["test"], two_dials(O["validation"], DATA["validation"])), DATA["test"])
    mr = metrics(apply_two(P["test"], DATA["test"], two_dials(P["validation"], DATA["validation"])), DATA["test"])
    san.append({"seed": seed, "run": run,
                "accuracy original": mo["accuracy"], "accuracy rerun": mr["accuracy"],
                "log loss original": mo["log loss"], "log loss rerun": mr["log loss"],
                "same top pick": float(np.mean([int(np.argmax(a)) == int(np.argmax(b))
                                                for a, b in zip(O["test"], P["test"])])),
                "gap within normal seed spread": bool(abs(mr["log loss"] - mo["log loss"]) <= spread.loc[run, "log loss"] + 1e-9)})
SANITY = pd.DataFrame(san)
print("\n1. SANITY: retrained vs original students, test split (2 dials)")
display(SANITY.round(4))
print(f"   {int(SANITY['gap within normal seed spread'].sum())} of {len(SANITY)} reruns are within the normal "
      f"seed-to-seed spread from Step 2")

# 2. Every student's SNIPS answers, 2 dials each (fitted on that student's own validation answers)
SNIPS_SEEDS = [0, 1, 2]
MV = [i for i, r in enumerate(DATA["test"]) if r["source"] == "mteb/amazon_massive_intent"]
MV_ROWS = [DATA["test"][i] for i in MV]
SN, rows = {}, []
for run in RUNS:
    for seed in SNIPS_SEEDS:
        src = ALL_PREDS[0, run] if seed == 0 else RER[seed, run]
        scores = SNIPS_SCORES[run] if seed == 0 else src["snips"]
        pooled = two_dials(src["validation"], DATA["validation"])
        SN[seed, run] = apply_two(scores, SNIPS, pooled)
        mv = metrics(apply_two([src["test"][i] for i in MV], MV_ROWS, pooled), MV_ROWS)
        rows.append({"seed": seed, "run": run, **{k: float(v) for k, v in metrics(SN[seed, run], SNIPS).items()},
                     "MASSIVE accuracy (seen)": float(mv["accuracy"]), "MASSIVE log loss (seen)": float(mv["log loss"])})
SN_PER_SEED = pd.DataFrame(rows)
def mean_range(x):
    return f"{x.mean():.3f}  ({x.min():.3f} to {x.max():.3f})"
COLS = ["accuracy", "ECE", "log loss", "MASSIVE accuracy (seen)", "MASSIVE log loss (seen)"]
print("\n2. SNIPS: MEAN (lowest to highest) across 3 seeds, 2 dials; MASSIVE (seen) alongside")
display(SN_PER_SEED.groupby("run").agg(**{c: (c, mean_range) for c in COLS}).reset_index())

# 3. Consistency: does each comparison hold in every seed? (ties shown as ties)
lines = []
for cand, base_run in COMPARISONS:
    for measure, lower_is_better in [("accuracy", False), ("ECE", True), ("log loss", True)]:
        gaps = []
        for seed in SNIPS_SEEDS:
            pick = lambda r: SN_PER_SEED[(SN_PER_SEED.seed == seed) & (SN_PER_SEED.run == r)][measure].iloc[0]
            c, o = pick(cand), pick(base_run)
            gaps.append((o - c) if lower_is_better else (c - o))       # positive = candidate better
        gaps = np.array(gaps)
        wins, ties = int((gaps > 0).sum()), int((gaps == 0).sum())
        lines.append({"comparison": f"{cand} vs {base_run}", "measure": measure,
                      "seeds where candidate better": f"{wins}/{len(gaps)}" + (f" ({ties} tied)" if ties else ""),
                      "average gap": round(float(gaps.mean()), 4),
                      "verdict": f"{cand} better in every seed" if wins == len(gaps) else
                                 (f"{cand} worse in every seed" if (gaps < 0).all() else "depends on the seed")})
SN_CONS = pd.DataFrame(lines)
print("\n3. CONSISTENCY on SNIPS: does each comparison hold in every seed?")
display(SN_CONS)

# 4. Ensembles: each run's three dialed students averaged, then a paired bootstrap over SNIPS questions
SN_ENS = {run: [np.mean([SN[s, run][i] for s in SNIPS_SEEDS], axis=0) for i in range(len(SNIPS))] for run in RUNS}
ens_rows = []
for run in RUNS:
    m = metrics(SN_ENS[run], SNIPS)
    singles = float(np.mean([metrics(SN[s, run], SNIPS)["log loss"] for s in SNIPS_SEEDS]))
    assert m["log loss"] <= singles + 1e-12, f"{run}: Jensen check failed"
    ens_rows.append({"run": run, "accuracy: ensemble": float(m["accuracy"]), "ECE: ensemble": float(m["ECE"]),
                     "log loss: ensemble": float(m["log loss"]), "log loss: one student (avg)": singles})
SN_ENS_TABLE = pd.DataFrame(ens_rows)
print("\n4. SNIPS ENSEMBLES (Jensen check passed for every run)")
display(SN_ENS_TABLE.round(3))

idx_all = np.random.default_rng(SNIPS_SEED).integers(0, len(SNIPS), size=(N_BOOT, len(SNIPS)))
stats = {run: per_row(SN_ENS[run], SNIPS) for run in RUNS}
def bvals(run, measure):
    conf, correct, nll = stats[run]
    if measure == "ECE":
        return np.array([ece_from(conf, correct, idx) for idx in idx_all])
    return np.array([(nll if measure == "log loss" else correct)[idx].mean() for idx in idx_all])
lines = []
for cand, base_run in COMPARISONS:
    for measure, lower_is_better in [("accuracy", False), ("ECE", True), ("log loss", True)]:
        c, o = bvals(cand, measure), bvals(base_run, measure)
        diff = (o - c) if lower_is_better else (c - o)
        lo, hi = np.percentile(diff, [2.5, 97.5])
        lines.append({"comparison": f"{cand} vs {base_run}", "measure": measure,
                      "candidate better in": f"{int((diff > 0).sum())}/{N_BOOT}",
                      "average gap": round(float(diff.mean()), 4), "95% range": f"[{lo:+.4f}, {hi:+.4f}]",
                      "verdict": f"{cand} better" if lo > 0 else (f"{cand} worse" if hi < 0 else "too close to call")})
SN_ENS_BOOT = pd.DataFrame(lines)
print("\nSNIPS ENSEMBLE vs ENSEMBLE, paired bootstrap")
display(SN_ENS_BOOT)

# 5. What held up under BOTH checks (every seed AND the ensemble bootstrap): the card's standard
def direction(v):
    return "better" if "better" in v else ("worse" if "worse" in v else None)
both = SN_CONS.merge(SN_ENS_BOOT, on=["comparison", "measure"], suffixes=("_seeds", "_ens"))
both["d1"], both["d2"] = both["verdict_seeds"].map(direction), both["verdict_ens"].map(direction)
ROBUST_SNIPS = both[both["d1"].notna() & (both["d1"] == both["d2"])]
print("\n5. HELD UP UNDER BOTH CHECKS on SNIPS:")
for _, r in ROBUST_SNIPS.iterrows():
    cand, base_run = r["comparison"].split(" vs ")
    print(f"   {cand} {r['d1']} than {base_run} on {r['measure']}")
if ROBUST_SNIPS.empty:
    print("   (none)")

SANITY.to_csv(f"{BASE_DIR}/step7_rerun_sanity.csv", index=False)
SN_PER_SEED.to_csv(f"{BASE_DIR}/step7_snips_per_seed.csv", index=False)
SN_CONS.to_csv(f"{BASE_DIR}/step7_snips_consistency.csv", index=False)
SN_ENS_TABLE.to_csv(f"{BASE_DIR}/step7_snips_ensembles.csv", index=False)
SN_ENS_BOOT.to_csv(f"{BASE_DIR}/step7_snips_ensemble_bootstrap.csv", index=False)

reruns found: seeds 1 and 2, all 6 runs

1. SANITY: retrained vs original students, test split (2 dials)


,seed,run,accuracy original,accuracy rerun,log loss original,log loss rerun,same top pick,gap within normal seed spread
0,1,A,0.9093,0.9188,0.2419,0.2366,0.9572,True
1,1,B,0.9132,0.9076,0.2853,0.2732,0.9477,True
2,1,C,0.8865,0.8865,0.3126,0.3199,0.9677,False
3,1,D,0.9093,0.9176,0.2561,0.2445,0.9599,True
4,1,E,0.9076,0.9060,0.2671,0.2622,0.9688,True
5,1,F,0.9176,0.9249,0.2494,0.2393,0.9622,False
6,2,A,0.9165,0.8971,0.2303,0.2703,0.9182,False
7,2,B,0.9232,0.9265,0.2683,0.2642,0.9683,True
8,2,C,0.8915,0.8948,0.3109,0.3077,0.9839,True
9,2,D,0.9171,0.9171,0.2485,0.2501,0.9766,True


   8 of 12 reruns are within the normal seed-to-seed spread from Step 2

2. SNIPS: MEAN (lowest to highest) across 3 seeds, 2 dials; MASSIVE (seen) alongside


,run,accuracy,ECE,log loss,MASSIVE accuracy (seen),MASSIVE log loss (seen)
0,A,0.949 (0.943 to 0.959),0.028 (0.027 to 0.028),0.155 (0.135 to 0.177),0.948 (0.936 to 0.960),0.164 (0.141 to 0.196)
1,B,0.948 (0.939 to 0.958),0.033 (0.031 to 0.034),0.175 (0.155 to 0.196),0.951 (0.946 to 0.956),0.170 (0.161 to 0.186)
2,C,0.951 (0.946 to 0.956),0.018 (0.010 to 0.027),0.152 (0.128 to 0.169),0.921 (0.906 to 0.929),0.211 (0.183 to 0.244)
3,D,0.953 (0.946 to 0.959),0.010 (0.008 to 0.013),0.137 (0.120 to 0.153),0.955 (0.949 to 0.963),0.135 (0.121 to 0.151)
4,E,0.960 (0.959 to 0.961),0.010 (0.004 to 0.015),0.125 (0.122 to 0.127),0.949 (0.946 to 0.953),0.147 (0.129 to 0.163)
5,F,0.950 (0.947 to 0.954),0.016 (0.011 to 0.022),0.139 (0.137 to 0.143),0.954 (0.946 to 0.966),0.138 (0.129 to 0.145)



3. CONSISTENCY on SNIPS: does each comparison hold in every seed?


,comparison,measure,seeds where candidate better,average gap,verdict
0,C vs A,accuracy,1/3 (1 tied),0.0019,depends on the seed
1,C vs A,ECE,3/3,0.0104,C better in every seed
2,C vs A,log loss,2/3,0.0031,depends on the seed
3,C vs B,accuracy,2/3,0.0032,depends on the seed
4,C vs B,ECE,3/3,0.0150,C better in every seed
5,C vs B,log loss,2/3,0.0223,depends on the seed
6,D vs A,accuracy,2/3 (1 tied),0.0038,depends on the seed
7,D vs A,ECE,3/3,0.0177,D better in every seed
8,D vs A,log loss,3/3,0.0183,D better in every seed
9,D vs B,accuracy,2/3,0.0051,depends on the seed



4. SNIPS ENSEMBLES (Jensen check passed for every run)


,run,accuracy: ensemble,ECE: ensemble,log loss: ensemble,log loss: one student (avg)
0,A,0.961,0.044,0.143,0.155
1,B,0.954,0.028,0.153,0.175
2,C,0.958,0.026,0.143,0.152
3,D,0.956,0.018,0.128,0.137
4,E,0.965,0.020,0.114,0.125
5,F,0.958,0.020,0.126,0.139



SNIPS ENSEMBLE vs ENSEMBLE, paired bootstrap


,comparison,measure,candidate better in,average gap,95% range,verdict
0,C vs A,accuracy,250/1000,-0.0029,"[-0.0133, +0.0076]",too close to call
1,C vs A,ECE,997/1000,0.0154,"[+0.0048, +0.0257]",C better
2,C vs A,log loss,512/1000,0.0001,"[-0.0164, +0.0154]",too close to call
3,C vs B,accuracy,777/1000,0.0040,"[-0.0067, +0.0143]",too close to call
4,C vs B,ECE,766/1000,0.0038,"[-0.0066, +0.0141]",too close to call
5,C vs B,log loss,868/1000,0.0108,"[-0.0070, +0.0289]",too close to call
6,D vs A,accuracy,124/1000,-0.0049,"[-0.0143, +0.0048]",too close to call
7,D vs A,ECE,998/1000,0.0236,"[+0.0123, +0.0344]",D better
8,D vs A,log loss,994/1000,0.0142,"[+0.0021, +0.0272]",D better
9,D vs B,accuracy,644/1000,0.0020,"[-0.0057, +0.0095]",too close to call



5. HELD UP UNDER BOTH CHECKS on SNIPS:
   C better than A on ECE
   D better than A on ECE
   D better than A on log loss
   D better than B on ECE
   D better than B on log loss
   E better than A on ECE
   E better than A on log loss
   E better than B on accuracy
   E better than B on ECE
   E better than B on log loss
   F better than A on ECE
   F better than B on log loss
   F worse than E on log loss
   E better than D on accuracy
   E better than C on log loss


## **[29] — Figures for the write-up**
Real reliability diagrams (A vs F vs the teacher) and validation curves across 3 seeds. No GPU.

In [ ]:
# Figures for the write-up, drawn from your real saved predictions.
# Needs: Cells 1-5, 13, 24 and 25 (Cell 25 builds the ensembles); for the SNIPS row also 26, 27, 28. No GPU.
import matplotlib.pyplot as plt
INK, MUTED, GRID = "#2C2C2A", "#6B6A65", "#E4E2DA"
PURPLE, TEAL, CORAL, AMBER, GRAY = "#7F77DD", "#1D9E75", "#D85A30", "#EF9F27", "#B4B2A9"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 13, "text.color": INK, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK, "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False, "xtick.color": MUTED, "ytick.color": MUTED,
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "savefig.dpi": 200, "savefig.bbox": "tight", "savefig.pad_inches": 0.25})
FIG_DIR = f"{BASE_DIR}/blog_figures"
os.makedirs(FIG_DIR, exist_ok=True)

# ---------- Figure 6: real reliability diagrams ----------
def reliability_panel(ax, probs, rows, title, color):
    conf, correct, _ = per_row(probs, rows)
    bins = np.minimum((conf * 10).astype(int), 9)
    for b in range(10):
        m = bins == b
        if m.sum() >= 5:                                   # skip near-empty bins: they're noise
            ax.bar((b + 0.5) / 10, correct[m].mean(), width=0.085, color=color, alpha=0.85, zorder=2)
            ax.text((b + 0.5) / 10, correct[m].mean() + 0.025, f"{int(m.sum())}", ha="center", fontsize=8.5, color=MUTED)
    ax.plot([0, 1], [0, 1], "--", color=MUTED, lw=1.2, zorder=3)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.08)
    ax.set_xticks([0, 0.5, 1.0]); ax.set_xticklabels(["0%", "50%", "100%"])
    ax.set_yticks([0, 0.5, 1.0]); ax.set_yticklabels(["0%", "50%", "100%"])
    ax.set_title(f"{title}\nECE {ece_from(conf, correct):.3f}", loc="left", fontsize=13)

teacher = {s: [np.array(r["teacher_probs"]) / np.sum(r["teacher_probs"]) for r in DATA[s]] for s in ("test", "heldout")}
TESTS = [("test", "familiar data")]
if "SN_ENS" in globals():                          # SNIPS ensembles, from Cell 28
    TESTS.append(("snips", "SNIPS"))
else:
    print("SNIPS row skipped: run Cells 26, 27 and 28 first to include it")
TESTS.append(("heldout", "PubMedQA"))
COLS = [("A", CORAL), ("E", PURPLE), ("F", TEAL), ("Teacher", GRAY)]
fig, axes = plt.subplots(len(TESTS), len(COLS), figsize=(16, 4.4 * len(TESTS)), sharex=True, sharey=True)
for row, (split, name) in enumerate(TESTS):
    rows = SNIPS if split == "snips" else DATA[split]
    for col, (who, color) in enumerate(COLS):
        ax = axes[row, col]
        if who == "Teacher":
            if split == "snips":
                ax.axis("off")
                ax.text(0.5, 0.5, "the teacher was\nnot run on SNIPS", ha="center", va="center",
                        fontsize=12, color=MUTED, transform=ax.transAxes)
                continue
            probs = teacher[split]
        else:
            probs = SN_ENS[who] if split == "snips" else ENS[who, split]
        label = "Teacher" if who == "Teacher" else f"{who}, ensemble"
        reliability_panel(ax, probs, rows, f"{label}: {name}", color)
for ax in axes[-1]:
    ax.set_xlabel("how sure it said it was")
for ax in axes[:, 0]:
    ax.set_ylabel("how often it was right")
fig.tight_layout(h_pad=2.5, w_pad=2)
fig.savefig(f"{FIG_DIR}/fig6_reliability_real.png"); plt.show()

# ---------- Figure 7: validation loss per epoch, 3 seeds ----------
fig, ax = plt.subplots(figsize=(9.5, 5.2))
style = {"A": (CORAL, 2.6), "F": (TEAL, 2.6), "E": (PURPLE, 2.0)}
ends = []
for run in RUNS:
    curves = np.array([[h["val_loss"] for h in ALL_PREDS[s, run]["history"]] for s in SEEDS_ALL])
    epochs = np.arange(1, curves.shape[1] + 1)
    color, lw = style.get(run, (GRAY, 1.4))
    ax.plot(epochs, curves.mean(0), color=color, lw=lw, marker="o", zorder=3 if run in style else 2)
    if run in style:
        ax.fill_between(epochs, curves.min(0), curves.max(0), color=color, alpha=0.12, zorder=1)
    ends.append((curves.mean(0)[-1], run, color))
# Labels at the right edge, nudged apart so close curves don't overlap
lo, hi = ax.get_ylim(); gap = (hi - lo) * 0.045
placed = []
for y, run, color in sorted(ends):
    y = max(y, placed[-1] + gap) if placed else y
    placed.append(y)
    ax.text(epochs[-1] + 0.06, y, run, color=color, fontsize=12, va="center",
            fontweight="bold" if run in style else "normal")
ax.set_xticks([1, 2, 3]); ax.set_xlabel("epoch"); ax.set_ylabel("validation log loss (lower is better)")
ax.set_title("Validation loss, mean of 3 seeds (shaded: lowest to highest seed)", loc="left", fontsize=13)
fig.savefig(f"{FIG_DIR}/fig7_validation_curves.png"); plt.show()
print("saved to", FIG_DIR)

# ---------- Figure 8: what training luck looks like (each seed is a dot) ----------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, (split, measure, title, base, base_label) in zip(axes, [
        ("test", "log loss", "Familiar data: log loss per seed", None, None),
        ("heldout", "accuracy", "PubMedQA: accuracy per seed", 0.552, "always answering \"yes\": 0.552")]):
    for i, run in enumerate(RUNS):
        vals = PER_SEED[(PER_SEED.split == split) & (PER_SEED.run == run)][measure].values
        ax.plot([vals.min(), vals.max()], [i, i], color=GRID, lw=6, solid_capstyle="round", zorder=1)
        ax.scatter(vals, [i] * len(vals), s=70, color=TEAL if run == "F" else PURPLE, zorder=3)
    if base is not None:
        ax.axvline(base, color=CORAL, ls="--", lw=1.5)
        ax.text(base + 0.001, -0.75, base_label, color=CORAL, fontsize=10.5, va="center")
    ax.set_yticks(range(len(RUNS))); ax.set_yticklabels(RUNS, fontsize=13, fontweight="bold")
    ax.set_ylim(len(RUNS) - 0.5, -1.1)                # top to bottom: A first, room for the label
    ax.set_xlabel(measure + (" (lower is better)" if measure == "log loss" else " (higher is better)"))
    ax.set_title(title, loc="left", fontsize=13)
fig.tight_layout(w_pad=4)
fig.savefig(f"{FIG_DIR}/fig8_seed_luck.png"); plt.show()
print("figure 8 saved")

## **[30] — Step 8: ModernBERT-large vs base**
Targets E and F, three tests, both checks; answers the question written before training. No GPU.

In [ ]:
# Step 8 analysis: ModernBERT-large vs base, for targets E and F.
# Needs: Cells 1-5, 13, 24, 25, 26, 27 and 28 (MODEL_SIZE in Cell 4 doesn't matter here). No GPU.
#
# PRIMARY QUESTION (written before training): on PubMedQA, does the large student beat the base student
# trained on the same target, on log loss, under BOTH checks, for E or F?
# SECONDARY: is the large student reliably WORSE anywhere (familiar data or SNIPS)?
LARGE_RUNS = ["E", "F"]
SPLITS = ["test", "snips", "heldout"]
SPLIT_NAME = {"test": "familiar", "snips": "SNIPS", "heldout": "PubMedQA"}
MEASURES = [("accuracy", False), ("ECE", True), ("log loss", True)]

# Redefine GRID locally to prevent collisions with the color string 'GRID' from Cell 29
GRID = np.exp(np.linspace(np.log(0.25), np.log(20.0), 200))
B_GRID = np.linspace(-8.0, 8.0, 321)

def rows_of(split):
    return SNIPS if split == "snips" else DATA[split]

LRG = {}
for seed in SEEDS_ALL:
    for run in LARGE_RUNS:
        path = f"{BASE_DIR}/large/seed{seed}/preds_{run}.pkl"
        assert os.path.exists(path), f"large run not finished: {path}"
        with open(path, "rb") as f:
            P = pickle.load(f)
        assert P.get("snips") is not None and len(P["snips"]) == len(SNIPS), \
            f"large seed {seed}, run {run}: no SNIPS answers saved (was Cell 26 run before training?)"
        LRG[seed, run] = P
print("large runs found:", sorted(LRG), "| kept epochs:", {k: v["best_epoch"] for k, v in sorted(LRG.items())})

# Every student's dialed chances on the three tests (2 dials, fitted on that student's own validation answers)
PROB = {}
for run in LARGE_RUNS:
    for seed in SEEDS_ALL:
        Pb = ALL_PREDS[seed, run]
        pooled = two_dials(Pb["validation"], DATA["validation"])
        PROB["base", seed, run, "test"] = apply_two(Pb["test"], DATA["test"], pooled)
        PROB["base", seed, run, "heldout"] = apply_two(Pb["heldout"], DATA["heldout"], pooled)
        PROB["base", seed, run, "snips"] = SN[seed, run]          # from Cell 28 (seeds 1-2: the retrained students)
        Pl = LRG[seed, run]
        pooled = two_dials(Pl["validation"], DATA["validation"])
        for split in SPLITS:
            PROB["large", seed, run, split] = apply_two(Pl[split], rows_of(split), pooled)

rows = []
for (size, seed, run, split), probs in PROB.items():
    rows.append({"size": size, "seed": seed, "run": run, "split": split,
                 **{k: float(v) for k, v in metrics(probs, rows_of(split)).items()}})
L_PER_SEED = pd.DataFrame(rows)

def mean_range(x):
    return f"{x.mean():.3f}  ({x.min():.3f} to {x.max():.3f})"
summary = (L_PER_SEED.groupby(["split", "run", "size"])[["accuracy", "ECE", "log loss"]]
           .agg(mean_range).reset_index())
summary["split"] = summary["split"].map(SPLIT_NAME)
print("\n1. SINGLE STUDENTS: mean (lowest to highest) over 3 seeds, 2 dials")
display(summary)

# 2. Every large seed against every base seed (9 pairings): seeds aren't shared across model sizes
lines = []
for run in LARGE_RUNS:
    for split in SPLITS:
        for measure, lower_is_better in MEASURES:
            sel = lambda size: L_PER_SEED[(L_PER_SEED["size"] == size) & (L_PER_SEED.run == run) &
                                          (L_PER_SEED.split == split)][measure].values
            gaps = np.array([(b - l) if lower_is_better else (l - b) for l in sel("large") for b in sel("base")])
            wins = int((gaps > 0).sum())
            lines.append({"run": run, "split": split, "measure": measure,
                          "pairings where large better": f"{wins}/{len(gaps)}",
                          "average gap": round(float(gaps.mean()), 4),
                          "verdict": "large better in all 9" if wins == len(gaps) else
                                     ("large worse in all 9" if (gaps < 0).all() else "depends on the seeds")})
L_CONS = pd.DataFrame(lines)
print("\n2. EVERY LARGE SEED vs EVERY BASE SEED (positive gap = large better)")
display(L_CONS.assign(split=L_CONS["split"].map(SPLIT_NAME)))

# 3. Ensembles (3 seeds averaged) and a paired bootstrap over questions
ENS8 = {(size, run, split): [np.mean([PROB[size, s, run, split][i] for s in SEEDS_ALL], axis=0)
                             for i in range(len(rows_of(split)))]
        for size in ("base", "large") for run in LARGE_RUNS for split in SPLITS}
ens_rows, boot_rows = [], []
for split in SPLITS:
    rows_ = rows_of(split)
    idx_all = np.random.default_rng(0).integers(0, len(rows_), size=(N_BOOT, len(rows_)))
    for run in LARGE_RUNS:
        m_b, m_l = metrics(ENS8["base", run, split], rows_), metrics(ENS8["large", run, split], rows_)
        ens_rows.append({"split": SPLIT_NAME[split], "run": run,
                         **{f"{k}: base": float(m_b[k]) for k in ("accuracy", "ECE", "log loss")},
                         **{f"{k}: large": float(m_l[k]) for k in ("accuracy", "ECE", "log loss")}})
        st = {size: per_row(ENS8[size, run, split], rows_) for size in ("base", "large")}
        def bv(size, measure):
            conf, correct, nll = st[size]
            if measure == "ECE":
                return np.array([ece_from(conf, correct, idx) for idx in idx_all])
            return np.array([(nll if measure == "log loss" else correct)[idx].mean() for idx in idx_all])
        for measure, lower_is_better in MEASURES:
            l, b = bv("large", measure), bv("base", measure)
            diff = (b - l) if lower_is_better else (l - b)
            lo, hi = np.percentile(diff, [2.5, 97.5])
            boot_rows.append({"run": run, "split": split, "measure": measure,
                              "large better in": f"{int((diff > 0).sum())}/{N_BOOT}",
                              "average gap": round(float(diff.mean()), 4), "95% range": f"[{lo:+.4f}, {hi:+.4f}]",
                              "verdict": "large better" if lo > 0 else ("large worse" if hi < 0 else "too close to call")})
L_ENS = pd.DataFrame(ens_rows)
L_BOOT = pd.DataFrame(boot_rows)
print("\n3. ENSEMBLES: base vs large")
display(L_ENS.round(3))
print("\nENSEMBLE BOOTSTRAP (positive gap = large better)")
display(L_BOOT.assign(split=L_BOOT["split"].map(SPLIT_NAME)))

# 4. Both checks, and the answers to the questions written before training
both = L_CONS.merge(L_BOOT, on=["run", "split", "measure"], suffixes=("_seeds", "_ens"))
def direction(v):
    return "better" if "better" in v else ("worse" if "worse" in v else None)
both["d1"], both["d2"] = both["verdict_seeds"].map(direction), both["verdict_ens"].map(direction)
L_ROBUST = both[both["d1"].notna() & (both["d1"] == both["d2"])]
print("\n4. HELD UP UNDER BOTH CHECKS (all 9 seed pairings AND the ensemble bootstrap):")
for _, r in L_ROBUST.iterrows():
    print(f"   {r['run']}: large {r['d1']} than base on {SPLIT_NAME[r['split']]} {r['measure']}")
if L_ROBUST.empty:
    print("   (none)")

primary = L_ROBUST[(L_ROBUST.split == "heldout") & (L_ROBUST.measure == "log loss") & (L_ROBUST.d1 == "better")]
worse = L_ROBUST[(L_ROBUST.d1 == "worse") & (L_ROBUST.split.isin(["test", "snips"]))]
freq = pd.Series([r["gold"] for r in DATA["heldout"]]).value_counts(normalize=True)
print("\nPRIMARY QUESTION: does large beat base on PubMedQA log loss under both checks?")
print("   YES, for " + ", ".join(primary["run"]) if len(primary) else
      "   NO: capacity alone did not close the PubMedQA gap (the next lever is training data)")
print("SECONDARY: is large reliably worse on familiar data or SNIPS?")
print("   " + ("; ".join(f"{r['run']} on {SPLIT_NAME[r['split']]} {r['measure']}" for _, r in worse.iterrows())
              if len(worse) else "no"))
print(f"For reference on PubMedQA: always answering '{freq.index[0]}' = accuracy {freq.iloc[0]:.3f}; "
      f"knowing only the label frequencies = log loss {float(-(freq * np.log(freq)).sum()):.3f}")

L_PER_SEED.to_csv(f"{BASE_DIR}/step8_large_per_seed.csv", index=False)
L_CONS.to_csv(f"{BASE_DIR}/step8_large_consistency.csv", index=False)
L_ENS.to_csv(f"{BASE_DIR}/step8_large_ensembles.csv", index=False)
L_BOOT.to_csv(f"{BASE_DIR}/step8_large_bootstrap.csv", index=False)

## **[31] — Step 9: LoRA vs full fine-tuning (target F)**
Large-LoRA F vs full large F (primary) and vs base F; both checks; answers the questions written before training.

In [ ]:
# Step 9 analysis: ModernBERT-large with LoRA (target F) vs full fine-tuning, and vs base.
# Needs: Cells 1-5, 13, 24, 25, 26, 27, 28 and 30 (Cell 30 loads the full fine-tuning runs). No GPU.
#
# PRIMARY (written before training): on PubMedQA, does LoRA-F beat full-fine-tuned large-F on log loss,
# under BOTH checks?
# SECONDARY: (a) does LoRA-F beat base-F on PubMedQA log loss under both checks?
#            (b) is LoRA-F reliably WORSE than full large-F on familiar data or SNIPS (the "learns less" cost)?
KINDS = {"base": "base, full fine-tuning", "large": "large, full fine-tuning", "lora": "large, LoRA"}
for seed in SEEDS_ALL:
    path = f"{BASE_DIR}/large_lora/seed{seed}/preds_F.pkl"
    assert os.path.exists(path), f"LoRA run not finished: {path}"
    with open(path, "rb") as f:
        P = pickle.load(f)
    assert P.get("snips") is not None and len(P["snips"]) == len(SNIPS), \
        f"LoRA seed {seed}: no SNIPS answers saved (was Cell 26 run before training?)"
    pooled = two_dials(P["validation"], DATA["validation"])
    for split in SPLITS:
        PROB["lora", seed, "F", split] = apply_two(P[split], rows_of(split), pooled)
    print(f"LoRA seed {seed}: kept epoch {P['best_epoch']}")

rows = [{"kind": k, "seed": s, "split": sp, **{m: float(v) for m, v in metrics(PROB[k, s, "F", sp], rows_of(sp)).items()}}
        for k in KINDS for s in SEEDS_ALL for sp in SPLITS]
F_PER_SEED = pd.DataFrame(rows)
tab = F_PER_SEED.groupby(["split", "kind"])[["accuracy", "ECE", "log loss"]].agg(mean_range).reset_index()
tab["split"], tab["kind"] = tab["split"].map(SPLIT_NAME), tab["kind"].map(KINDS)
print("\n1. TARGET F, SINGLE STUDENTS: mean (lowest to highest) over 3 seeds, 2 dials")
display(tab)

F_ENS = {(k, sp): [np.mean([PROB[k, s, "F", sp][i] for s in SEEDS_ALL], axis=0) for i in range(len(rows_of(sp)))]
         for k in KINDS for sp in SPLITS}
ens_tab = pd.DataFrame([{"split": SPLIT_NAME[sp], "kind": KINDS[k],
                         **{m: float(v) for m, v in metrics(F_ENS[k, sp], rows_of(sp)).items()}}
                        for sp in SPLITS for k in KINDS])
print("\n2. TARGET F, ENSEMBLES")
display(ens_tab.round(3))

def compare(cand, ref):
    """Both checks: all 9 seed pairings, and a paired bootstrap over questions for the ensembles."""
    out = []
    for split in SPLITS:
        rows_ = rows_of(split)
        idx_all = np.random.default_rng(0).integers(0, len(rows_), size=(N_BOOT, len(rows_)))
        st = {k: per_row(F_ENS[k, split], rows_) for k in (cand, ref)}
        def bv(k, measure):
            conf, correct, nll = st[k]
            if measure == "ECE":
                return np.array([ece_from(conf, correct, idx) for idx in idx_all])
            return np.array([(nll if measure == "log loss" else correct)[idx].mean() for idx in idx_all])
        for measure, lower_is_better in MEASURES:
            sel = lambda k: F_PER_SEED[(F_PER_SEED.kind == k) & (F_PER_SEED.split == split)][measure].values
            gaps = np.array([(r - c) if lower_is_better else (c - r) for c in sel(cand) for r in sel(ref)])
            wins = int((gaps > 0).sum())
            v_seeds = ("better" if wins == len(gaps) else "worse" if (gaps < 0).all() else None)
            c, r = bv(cand, measure), bv(ref, measure)
            diff = (r - c) if lower_is_better else (c - r)
            lo, hi = np.percentile(diff, [2.5, 97.5])
            v_ens = "better" if lo > 0 else ("worse" if hi < 0 else None)
            out.append({"comparison": f"{KINDS[cand]} vs {KINDS[ref]}", "split": split, "measure": measure,
                        "seed pairings where better": f"{wins}/{len(gaps)}",
                        "ensemble bootstrap better in": f"{int((diff > 0).sum())}/{N_BOOT}",
                        "average gap (ensembles)": round(float(diff.mean()), 4),
                        "both checks": v_seeds if (v_seeds and v_seeds == v_ens) else "not settled"})
    return pd.DataFrame(out)

VS_LARGE, VS_BASE = compare("lora", "large"), compare("lora", "base")
for title, df in [("3. LoRA vs FULL FINE-TUNING (both large)", VS_LARGE), ("4. LoRA (large) vs BASE", VS_BASE)]:
    print(f"\n{title}  (positive gap = LoRA better)")
    display(df.assign(split=df["split"].map(SPLIT_NAME)))

def settled(df, split, measure, way):
    r = df[(df.split == split) & (df.measure == measure)]
    return bool(len(r)) and r["both checks"].iloc[0] == way
cost = VS_LARGE[(VS_LARGE["both checks"] == "worse") & VS_LARGE.split.isin(["test", "snips"])]
print("\nPRIMARY: does LoRA-F beat full-fine-tuned large-F on PubMedQA log loss, under both checks?")
print("   YES: support for the 'full fine-tuning overwrote general ability' explanation"
      if settled(VS_LARGE, "heldout", "log loss", "better") else
      "   NO: this LoRA setup doesn't support the forgetting explanation; the conclusion stays 'data, not capacity'")
print("SECONDARY (a): does LoRA-F beat base-F on PubMedQA log loss, under both checks?")
print("   " + ("yes" if settled(VS_BASE, "heldout", "log loss", "better") else
               "no, base F is reliably better" if settled(VS_BASE, "heldout", "log loss", "worse") else "not settled"))
print("SECONDARY (b): is LoRA-F reliably worse than full large-F on familiar data or SNIPS?")
print("   " + ("; ".join(f"{SPLIT_NAME[r.split]} {r.measure}" for r in cost.itertuples()) if len(cost) else "no"))

F_PER_SEED.to_csv(f"{BASE_DIR}/step9_lora_per_seed.csv", index=False)
ens_tab.to_csv(f"{BASE_DIR}/step9_lora_ensembles.csv", index=False)
pd.concat([VS_LARGE, VS_BASE]).to_csv(f"{BASE_DIR}/step9_lora_comparisons.csv", index=False)

## **[32] — Sprint: the E controls and the weighting sweep**
G (E shuffled), H (E flattened), W25 and W75 against A, B, E and F; both checks; answers the pre-registered questions.

In [27]:
# Sprint analysis: the E controls (G, H) and the weighting sweep (W25, W75), against A, B, E and F.
# Needs: Cells 1-5, 13, 24, 25, 26, 27 and 28. No GPU.
#
# PRIMARY (written before training; amended before training, see the report): on SNIPS log loss, is E reliably
# better than BOTH G (E shuffled: same softness, random pattern) AND H (E flattened: no pattern, but softer)?
#   both  -> the teacher's knowledge of WHICH options are confusable matters
#   G only -> G's random pattern hurts, but the knowledge itself isn't shown to help
#   neither -> softness and per-question confidence explain E's lead
# SECONDARY: H vs B (per-question confidence vs fixed smoothing); G vs H (does a random pattern hurt?)
SPRINT_CANDIDATES = ["G", "H", "W25", "W75"]
BASE_RUNS = ["A", "B", "E", "F"]
SPLITS = ["test", "snips", "heldout"]
SPLIT_NAME = {"test": "familiar", "snips": "SNIPS", "heldout": "PubMedQA"}
MEASURES = [("accuracy", False), ("ECE", True), ("log loss", True)]
rows_of = lambda split: SNIPS if split == "snips" else DATA[split]

NEW = {}                                         # sprint runs found, every seed
for run in SPRINT_CANDIDATES:
    found = {}
    for seed in SEEDS_ALL:
        path = f"{seed_dir(seed)}/preds_{run}.pkl"
        if os.path.exists(path):
            with open(path, "rb") as f:
                found[seed] = pickle.load(f)
    if len(found) == len(SEEDS_ALL):
        for seed, P in found.items():
            assert P.get("snips") is not None and len(P["snips"]) == len(SNIPS), \
                f"{run} seed {seed}: no SNIPS answers saved (was Cell 26 run before training?)"
        NEW[run] = found
    elif found:
        print(f"{run}: only seeds {sorted(found)} finished; left out until all {len(SEEDS_ALL)} are done")
SPRINT_DONE = list(NEW)
assert SPRINT_DONE, "no sprint run has all three seeds yet"
print("sprint runs found:", SPRINT_DONE)

# Every student's dialed chances (2 dials, fitted on its own validation answers)
PR = {}
for run in BASE_RUNS:
    for seed in SEEDS_ALL:
        P = ALL_PREDS[seed, run]
        pooled = two_dials(P["validation"], DATA["validation"])
        PR[run, seed, "test"] = apply_two(P["test"], DATA["test"], pooled)
        PR[run, seed, "heldout"] = apply_two(P["heldout"], DATA["heldout"], pooled)
        PR[run, seed, "snips"] = SN[seed, run]               # from Cell 28 (seeds 1-2: the retrained students)
for run, seeds in NEW.items():
    for seed, P in seeds.items():
        pooled = two_dials(P["validation"], DATA["validation"])
        for split in SPLITS:
            PR[run, seed, split] = apply_two(P[split], rows_of(split), pooled)
ALL_RUNS = BASE_RUNS + SPRINT_DONE

per_seed = pd.DataFrame([{"run": r, "seed": s, "split": sp, **{k: float(v) for k, v in metrics(PR[r, s, sp], rows_of(sp)).items()}}
                         for r in ALL_RUNS for s in SEEDS_ALL for sp in SPLITS])
mr = lambda x: f"{x.mean():.3f} ({x.min():.3f} to {x.max():.3f})"
tab = per_seed.groupby(["split", "run"])[["accuracy", "ECE", "log loss"]].agg(mr).reset_index()
tab["split"] = tab["split"].map(SPLIT_NAME)
print("\n1. SINGLE STUDENTS: mean (lowest to highest) over 3 seeds, 2 dials")
display(tab)

# Comparisons: each one isolates one ingredient (see the ladder in the write-up)
CMP = [("E", "G"), ("E", "H"), ("G", "H"), ("H", "B"), ("G", "B"), ("G", "A"), ("H", "A"),
       ("W25", "E"), ("W25", "F"), ("W75", "F"), ("W75", "A")]
CMP = [(c, b) for c, b in CMP if c in ALL_RUNS and b in ALL_RUNS]
ENSP = {(r, sp): [np.mean([PR[r, s, sp][i] for s in SEEDS_ALL], axis=0) for i in range(len(rows_of(sp)))]
        for r in ALL_RUNS for sp in SPLITS}
lines = []
for split in SPLITS:
    rows_ = rows_of(split)
    idx_all = np.random.default_rng(0).integers(0, len(rows_), size=(N_BOOT, len(rows_)))
    stats = {r: per_row(ENSP[r, split], rows_) for r in ALL_RUNS}
    def bv(r, m):
        conf, correct, nll = stats[r]
        if m == "ECE":
            return np.array([ece_from(conf, correct, idx) for idx in idx_all])
        return np.array([(nll if m == "log loss" else correct)[idx].mean() for idx in idx_all])
    for cand, base in CMP:
        for m, lower in MEASURES:
            pick = lambda r: per_seed[(per_seed.run == r) & (per_seed.split == split)].sort_values("seed")[m].values
            gaps = (pick(base) - pick(cand)) if lower else (pick(cand) - pick(base))   # paired by seed
            seeds_v = "better" if (gaps > 0).all() else ("worse" if (gaps < 0).all() else None)
            diff = (bv(base, m) - bv(cand, m)) if lower else (bv(cand, m) - bv(base, m))
            lo, hi = np.percentile(diff, [2.5, 97.5])
            ens_v = "better" if lo > 0 else ("worse" if hi < 0 else None)
            lines.append({"split": split, "comparison": f"{cand} vs {base}", "measure": m,
                          "seeds where candidate better": f"{int((gaps > 0).sum())}/{len(gaps)}",
                          "ensemble bootstrap better in": f"{int((diff > 0).sum())}/{N_BOOT}",
                          "average gap (ensembles)": round(float(diff.mean()), 4),
                          "both checks": seeds_v if (seeds_v and seeds_v == ens_v) else "not settled"})
SPRINT_CMP = pd.DataFrame(lines)
print("\n2. COMPARISONS (positive gap = candidate better; seeds paired, since they share batch order and shuffles)")
display(SPRINT_CMP.assign(split=SPRINT_CMP["split"].map(SPLIT_NAME)))

def verdict(cand, base, split="snips", m="log loss"):
    r = SPRINT_CMP[(SPRINT_CMP.comparison == f"{cand} vs {base}") & (SPRINT_CMP.split == split) & (SPRINT_CMP.measure == m)]
    return r["both checks"].iloc[0] if len(r) else "not run"
vg, vh = verdict("E", "G"), verdict("E", "H")
print(f"\nPRIMARY: is E reliably better than BOTH G and H on SNIPS log loss?  (E vs G: {vg} | E vs H: {vh})")
if "not run" in (vg, vh):
    print("   G and H must both be trained first")
elif vg == "better" and vh == "better":
    print("   YES: E beats an equally soft target AND a pattern-free one: the teacher's knowledge matters")
elif vg == "better":
    print("   PARTLY: E beats G but not H. G's random pattern hurts; the knowledge itself isn't shown to help")
elif "worse" in (vg, vh):
    print("   NO, and a control is reliably BETTER than E: softness, not knowledge, explains E's lead")
else:
    print("   NO: E beats neither under both checks; softness and per-question confidence explain its lead")
print("SECONDARY: H vs B (per-question confidence vs fixed smoothing):", verdict("H", "B"))
print("SECONDARY: G vs H (does a random pattern hurt?):", verdict("G", "H"),
      "(\"worse\" means the random pattern hurts)")

per_seed.to_csv(f"{BASE_DIR}/step10_sprint_per_seed.csv", index=False)
SPRINT_CMP.to_csv(f"{BASE_DIR}/step10_sprint_comparisons.csv", index=False)

sprint runs found: ['G', 'H', 'W25', 'W75']

1. SINGLE STUDENTS: mean (lowest to highest) over 3 seeds, 2 dials


,split,run,accuracy,ECE,log loss
0,PubMedQA,A,0.580 (0.538 to 0.614),0.130 (0.097 to 0.171),0.970 (0.945 to 0.997)
1,PubMedQA,B,0.579 (0.568 to 0.585),0.092 (0.020 to 0.149),0.987 (0.957 to 1.005)
2,PubMedQA,E,0.596 (0.591 to 0.605),0.086 (0.048 to 0.107),0.935 (0.924 to 0.946)
3,PubMedQA,F,0.595 (0.584 to 0.607),0.046 (0.019 to 0.068),0.914 (0.888 to 0.953)
4,PubMedQA,G,0.578 (0.566 to 0.594),0.062 (0.052 to 0.081),0.939 (0.919 to 0.963)
5,PubMedQA,H,0.594 (0.589 to 0.600),0.053 (0.032 to 0.067),0.917 (0.895 to 0.938)
6,PubMedQA,W25,0.580 (0.573 to 0.588),0.082 (0.071 to 0.094),0.941 (0.929 to 0.956)
7,PubMedQA,W75,0.568 (0.558 to 0.573),0.081 (0.027 to 0.133),0.970 (0.925 to 0.995)
8,SNIPS,A,0.949 (0.943 to 0.959),0.028 (0.027 to 0.028),0.155 (0.135 to 0.177)
9,SNIPS,B,0.948 (0.939 to 0.958),0.033 (0.031 to 0.034),0.175 (0.155 to 0.196)



2. COMPARISONS (positive gap = candidate better; seeds paired, since they share batch order and shuffles)


,split,comparison,measure,seeds where candidate better,ensemble bootstrap better in,average gap (ensembles),both checks
0,familiar,E vs G,accuracy,1/3,35/1000,-0.0062,not settled
1,familiar,E vs G,ECE,3/3,624/1000,0.0017,not settled
2,familiar,E vs G,log loss,1/3,69/1000,-0.0071,not settled
3,familiar,E vs H,accuracy,0/3,85/1000,-0.0041,not settled
4,familiar,E vs H,ECE,2/3,329/1000,-0.0025,not settled
...,...,...,...,...,...,...,...
94,PubMedQA,W75 vs F,ECE,1/3,177/1000,-0.0232,not settled
95,PubMedQA,W75 vs F,log loss,0/3,0/1000,-0.0500,worse
96,PubMedQA,W75 vs A,accuracy,1/3,84/1000,-0.0258,not settled
97,PubMedQA,W75 vs A,ECE,2/3,999/1000,0.0746,not settled



PRIMARY: is E reliably better than BOTH G and H on SNIPS log loss?  (E vs G: better | E vs H: not settled)
   PARTLY: E beats G but not H. G's random pattern hurts; the knowledge itself isn't shown to help
SECONDARY: H vs B (per-question confidence vs fixed smoothing): better
SECONDARY: G vs H (does a random pattern hurt?): not settled ("worse" means the random pattern hurts)


In [28]:
print(SPRINT_CMP.assign(split=SPRINT_CMP["split"].map(SPLIT_NAME)).to_string(index=False))

   split comparison  measure seeds where candidate better ensemble bootstrap better in  average gap (ensembles) both checks
familiar     E vs G accuracy                          1/3                      35/1000                  -0.0062 not settled
familiar     E vs G      ECE                          3/3                     624/1000                   0.0017 not settled
familiar     E vs G log loss                          1/3                      69/1000                  -0.0071 not settled
familiar     E vs H accuracy                          0/3                      85/1000                  -0.0041 not settled
familiar     E vs H      ECE                          2/3                     329/1000                  -0.0025 not settled
familiar     E vs H log loss                          1/3                     579/1000                   0.0010 not settled
familiar     G vs H accuracy                          1/3                     751/1000                   0.0022 not settled
familiar